# Lachua — raw XRF + grain size in, all figures out (v2)

**What this notebook does, in order**

| Cell | What it makes | Main settings you can change |
|---|---|---|
| 0 | clears old uploads / figures | – |
| 1 | reads your raw XRF + Malvern grain-size files, computes clr and log-ratios | `CLR_GROUPS`, `RATIOS`, `FILES` |
| 1b | *only if a file fails to load*: shows what the file looks like | `LOOK_AT` |
| 2 | events, ages, colours, drawing helpers | `MIN_SAMPLES`, `DEPTH_LIM`, `COLOR_BY`, `BREAK_AT_GAPS` |
| 3 | **Fig 1** – Mean grain size → Mn → lithogenic → authigenic (clr) | `SHOW_SORTING` |
| 4 | **Fig 2** – Mean grain size → ln(Mn/Fe) → ln(Ti/Ca) → ln(Zr/K) | `RATIOS` (Cell 1) |
| 5 | PCA of all samples (biplot + PC scores downcore) | `SHOW` |
| 6 | ln(Zr/K) vs measured grain size, with r | `GS_MATCH_TOL` |
| 8a–8e | **your MATLAB `PCA_Event_script.m`, in Python** (Figs 01–16) | `PCA_CORES`, `EV_MIN_SAMPLES`, `N_PERMS` |
| 9 | **Bertrand et al. (2024) Fig. 12** – LDA biplot with your events | `F12_*` |
| 10 | Bertrand et al. (2024) Fig. 20A – *waiting for the paper* | – |
| 7 | QC checks – run before using any figure | – |
| 11 | download everything | – |

**How to read the code.** Every cell starts with a box that says what it does.
Lines starting with `#` are plain-English notes and are never run. Settings
(the things you are meant to change) are always at the **top** of a cell in
CAPITALS, e.g. `MIN_SAMPLES = 4`.

**Reference for element groups and ratios:** Bertrand, S. et al. (2024)
*Inorganic geochemistry of lake sediments: a review of analytical techniques
and guidelines for data interpretation.* Earth-Science Reviews 249, 104639.

**Grain-size statistics:** Folk, R.L. & Ward, W.C. (1957) *Brazos River bar: a
study in the significance of grain size parameters.* Journal of Sedimentary
Petrology 27, 3–26. Graphic mean (Mz), inclusive graphic standard deviation
(sorting), inclusive graphic skewness and graphic kurtosis, with their verbal
classes (e.g. *poorly sorted*, *fine skewed*, *mesokurtic*).

**Grain-size class names:** Udden, J.A. (1914) *Mechanical composition of
clastic sediments.* GSA Bulletin 25, 655–744; Wentworth, C.K. (1922) *A scale
of grade and class terms for clastic sediments.* Journal of Geology 30,
377–392. Clay < 3.9 µm, silt 3.9–62.5 µm, sand 62.5–2000 µm, each split into
1-φ classes (very fine … coarse silt, very fine … very coarse sand).

## How to read the figures – a short guide

**1. The φ (phi) scale – why grain size has two units.**
Folk & Ward (1957) do their maths on the φ scale, φ = −log₂(size in mm).
Each step of 1 φ halves the grain size, and **larger φ means FINER grain**:

| µm | 1000 | 500 | 250 | 125 | 62.5 | 31.25 | 15.6 | 7.8 | 3.9 |
|---|---|---|---|---|---|---|---|---|---|
| φ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |

Programs such as GRADISTAT (Blott & Pye 2001) report the Folk & Ward mean in φ.
This notebook converts every size to **µm** (µm = 1000 × 2^−φ), so a larger
number always means a coarser grain. **Sorting stays in φ**: it is a spread
(a width on the φ scale), not a size. See Folk & Ward (1957), and Blott & Pye
(2001, *Earth Surface Processes and Landforms* 26, 1237–1248).

**2. The grey bands behind mean grain size** are the Udden–Wentworth classes
(Udden 1914; Wentworth 1922): clay < 3.9 µm, very fine silt 3.9–7.8, fine silt
7.8–15.6, medium silt 15.6–31, coarse silt 31–62.5, then sands. A point sitting
in the "coarse silt" band has a mean size of coarse silt.

**3. clr values (Fig 1).** XRF counts are not concentrations. They depend on
water content, grain size and the scanner, and the elements are read together
as parts of one whole. The centred log-ratio, clr(x) = ln(x / geometric mean of
all elements), removes this "closed-sum" effect (Aitchison 1986; Weltje &
Tjallingii 2008; Bertrand et al. 2024, Sect. on log-ratio transformation).
**0 = average for that sample**, positive = richer than average, and
the units are "natural-log units". So +0.7 means about 2× the average.

**4. Log-ratios (Fig 2).** ln(A/B) compares two elements directly. It is 0 when
A and B are equal, +0.69 when A is twice B, and −0.69 when A is half of B.
Always read the *changes* downcore, not the absolute value.
(Weltje & Tjallingii 2008, *EPSL* 274, 423–438; Bertrand et al. 2024.)

**5. PCA and LDA (Cells 5, 8, 9).** PCA finds the directions in which samples
differ most, without knowing the events. LDA is told which samples belong to
which event and finds the directions that separate the events best. Arrows =
which variable pulls samples which way (Davis 2002, *Statistics and Data
Analysis in Geology*, Wiley).

In [ ]:
# =============================================================================
# CELL 0 - housekeeping. RUN THIS FIRST whenever you re-upload data.
#          Colab does not overwrite: uploading GUAC_29A_GSD.xlsx a second time
#          leaves BOTH GUAC_29A_GSD.xlsx and GUAC_29A_GSD (1).xlsx in /content.
#          This clears the duplicates and the old figures.
# =============================================================================
!rm -f /content/*.png
!rm -f /content/*\ \(1\).* /content/*\ \(2\).* /content/*\ \(3\).*
!ls -la /content

## Cell 1 – load data, compute clr and log-ratios
Upload the raw XRF counts and the **cleaned** Malvern grain-size files (with the
fake points removed). Rows you deleted – or cells you left empty – are simply
treated as *no measurement*; nothing is interpolated to fill them.

**Change here:** `CLR_GROUPS` (which elements, in which group, in which order)
and `RATIOS` (which log-ratios). Every figure below follows these two lists.

**Raw Malvern files** (e.g. `GUAC_29A_RawMasterGSD.xlsx`, one column per size
bin): D10, D50, D90 and the **Folk & Ward (1957)** mean, sorting, skewness
and kurtosis are **calculated here** from the bins, together with their Folk &
Ward class names, the **Udden–Wentworth** class of the mean, and the % of the
sample in every Udden–Wentworth class (clay, very fine silt … very coarse sand).
The mean grain-size panels show the Udden–Wentworth classes as grey bands
(`WENTWORTH_BANDS` in Cell 2 turns them off).

**Units are checked automatically.** A mean (or D-value) column given in φ, e.g.
GRADISTAT's Folk & Ward output, is converted to µm, and Cell 1 prints what it
detected (`units detected: ... <-- CHECK THESE`). Mean grain-size axes are plain
µm from 0 with a tick every `GS_TICK_UM` (10), in a panel `GS_PANEL_CM` (4.5 cm)
wide. Both settings are in Cell 2. Repeat runs of the
same sample are averaged first. The results are saved as
`GS_STATS_<core>.csv`. If the file also has Malvern's own Dx(50), Cell 1 prints
how well the two agree. If the difference is more than a few %, change `BIN_EDGE`.

File names are matched by **words** (`'GUAC 29A GSD'`), so renaming a file is
fine as long as those words stay in the name.

In [ ]:
# =============================================================================
# CELL 1 - UPLOAD RAW XRF + GRAIN SIZE; compute clr and log-ratios in Python
# Recognised names:  ...29A...raw...  /  stats_29A...  (same for 24A)
# =============================================================================
import pandas as pd, numpy as np, os, glob, re
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.ticker import MultipleLocator, MaxNLocator, NullFormatter
from matplotlib.lines import Line2D
from google.colab import files

plt.rcParams['axes.grid'] = False

# =============================================================================
# SETTINGS YOU WILL MOST LIKELY WANT TO CHANGE
# =============================================================================

# ---- 1. clr elements, grouped the way Bertrand et al. (2024) groups them ----
# The clr figure draws the panels LEFT TO RIGHT in exactly this order:
#     Mean grain size | Mn | lithogenic elements | authigenic elements
# Each group gets a bracket and a name above its panels.
# To move an element to another group, cut it from one list and paste it in
# the other. To drop an element, delete it. Every figure follows this list.
CLR_GROUPS = [
    # Mn on its own: it moves with oxygen at the lake floor (redox), not with
    # the catchment, so it is read differently from everything else.
    ('Redox',                 ['Mn']),

    # Lithogenic = washed in from the catchment as mineral grains.
    #   Al, K, Ti -> clays / feldspars / Ti-oxides (fine detritus)
    #   Zr        -> zircon, a heavy mineral in the coarser silt/sand
    #   Si        -> quartz and silicates. NOTE: diatoms also make Si
    #                (biogenic silica). If Si does not follow Ti/Al, move it.
    #   Fe        -> mostly detrital, BUT it is also redox sensitive; that is
    #                why ln(Mn/Fe) is used as the redox ratio below.
    ('Lithogenic (detrital)', ['Al', 'Si', 'K', 'Ti', 'Fe', 'Zr']),

    # Authigenic = formed inside the lake.
    #   Ca, Sr -> calcite / carbonate. Lachua sits on karst, so some Ca can
    #             also be detrital limestone - check it against Ti.
    #   S      -> sulfides and organic matter (low-oxygen conditions)
    ('Authigenic',            ['Ca', 'Sr', 'S']),
]
ELS = [el for group_name, group_elements in CLR_GROUPS for el in group_elements]

# ---- 2. log-ratios (natural log, as in Bertrand et al. 2024) ----------------
# Each line is: (numerator, denominator, panel label, colour).
# The ratio figure draws: Mean grain size | these ratios, in this order.
RATIOS = [
    ('Mn', 'Fe', 'ln(Mn/Fe)', '#38a169'),   # redox: higher = more oxygen at the lake floor
    ('Ti', 'Ca', 'ln(Ti/Ca)', '#2c7a7b'),   # detrital input vs carbonate
    ('Zr', 'K',  'ln(Zr/K)',  '#b83280'),   # grain size: coarse (zircon) vs fine (clay)
]
GS_RATIO = ('Zr', 'K')         # ratio tested against measured grain size in Cell 6

print("Upload: raw XRF (both cores) and grain-size stats (both cores).")
try: files.upload()
except Exception: pass
print("\nIn /content:")
for f in sorted(glob.glob('/content/*')):
    if not os.path.isdir(f): print("  ", os.path.basename(f))

DATA_EXT = ('.csv', '.xlsx', '.xls')
EV_ALIASES_LIGHT = ('event','eventid','eventno','eventnumber')

def _match(cols, keys):
    for c in cols:
        n = _norm(c)
        if any(n == k for k in keys): return c
    for c in cols:
        n = _norm(c)
        if any(n.startswith(k) for k in keys): return c
    return None

# EXACT files to use. Give the name without the extension; matching ignores
# case, spaces, hyphens and underscores, so a partial name is enough.
# Set a slot to None to fall back to content-based auto-detection.
# Matching is by WORDS: 'GUAC 29A GSD' finds 'GUAC_29A_GSD.xlsx' AND
# 'GUAC_29A_RawMasterGSD.xlsx', because all three words are in both names.
# If nothing matches, the notebook does not stop - it looks at every uploaded
# file with the core tag in its name and decides from the columns inside.
FILES = {'GUAC-29A': dict(xrf='GUAC 29A RepoOG', gs='GUAC 29A GSD'),
         'GUAC-24A': dict(xrf='GUAC 24A RepoOG', gs='GUAC 24A GSD')}

# ---- raw Malvern grain-size files -------------------------------------------
GS_SOURCE = 'auto'   # 'auto'    = if the file has raw size-bin columns, calculate
                     #             everything from the bins (recommended for raw
                     #             master files); otherwise use its D10/D50/D90
                     # 'bins'    = always calculate from the bins
                     # 'columns' = always use the file's own D10/D50/D90 columns
BIN_EDGE  = 'lower'  # what the size in a bin's column header means:
                     # 'lower' = lower edge of the size class (Malvern export)
                     # 'upper' = upper edge   'mid' = centre of the class
                     # Unsure? Run once, then look at the "check vs Malvern D50"
                     # line printed in Cell 1: pick the setting that gives the
                     # smallest difference.

# ---- grain-size references ----------------------------------------------------
# Statistics : Folk, R.L. & Ward, W.C. (1957) Brazos River bar: a study in the
#              significance of grain size parameters. J. Sediment. Petrol. 27, 3-26.
# Class names: Udden, J.A. (1914) Mechanical composition of clastic sediments.
#              GSA Bulletin 25, 655-744.  Wentworth, C.K. (1922) A scale of grade
#              and class terms for clastic sediments. J. Geology 30, 377-392.
#
# Udden-Wentworth classes: each boundary is a power of 2 in mm, so each class
# is exactly 1 phi wide.  (name, lower limit µm, upper limit µm)
WENTWORTH = [
    ('clay',             0.0,      3.90625),   # < 1/256 mm (9 phi and finer)
    ('very fine silt',   3.90625,  7.8125),    # 1/256 - 1/128 mm   (8-9 phi)
    ('fine silt',        7.8125,   15.625),    # 1/128 - 1/64 mm    (7-8 phi)
    ('medium silt',      15.625,   31.25),     # 1/64  - 1/32 mm    (6-7 phi)
    ('coarse silt',      31.25,    62.5),      # 1/32  - 1/16 mm    (5-6 phi)
    ('very fine sand',   62.5,     125.0),     # 1/16  - 1/8 mm     (4-5 phi)
    ('fine sand',        125.0,    250.0),     # 1/8   - 1/4 mm     (3-4 phi)
    ('medium sand',      250.0,    500.0),     # 1/4   - 1/2 mm     (2-3 phi)
    ('coarse sand',      500.0,    1000.0),    # 1/2   - 1 mm       (1-2 phi)
    ('very coarse sand', 1000.0,   2000.0),    # 1     - 2 mm       (0-1 phi)
    ('granule',          2000.0,   4000.0),    # 2     - 4 mm       (-1-0 phi)
]
CLAY_SILT_SAND = (3.90625, 62.5)   # µm: clay | silt | sand limits (Wentworth 1922)

def read_any(p):
    return pd.read_excel(p) if p.lower().endswith(('.xlsx','.xls')) \
           else pd.read_csv(p, encoding='utf-8-sig')

def _sheets(p):
    """Every sheet, read with NO header row assumed."""
    if p.lower().endswith(('.xlsx','.xls')):
        return pd.read_excel(p, sheet_name=None, header=None)
    return {'csv': pd.read_csv(p, header=None, encoding='utf-8-sig',
                               on_bad_lines='skip')}

def _rising_numbers(vals):
    v = [_size(x) for x in vals]
    v = [x for x in v if x is not None]
    return len(v) >= 10 and all(y > x for x, y in zip(v, v[1:]))

def _is_depthy(vals):
    return any(k in str(x).lower() for x in vals
               for k in ('depth','interval','sample name','sample id'))

def find_header(raw, look=15):
    """(header_row, transposed). Finds the real header in exports that carry a
    title row, and spots sheets written with samples down the columns."""
    for i in range(min(look, len(raw))):
        if _is_depthy(raw.iloc[i].tolist()): return i, False
    for i in range(min(look, len(raw))):
        if _rising_numbers(raw.iloc[i].tolist()): return i, False
    for j in range(min(look, raw.shape[1])):
        col = raw.iloc[:, j].tolist()
        if _is_depthy(col[:look]) or _rising_numbers(col): return j, True
    return 0, False

def _assemble(raw, hr, tr):
    r = raw.T.reset_index(drop=True) if tr else raw.reset_index(drop=True)
    hdr = r.iloc[hr].tolist()
    out = r.iloc[hr+1:].reset_index(drop=True)
    out.columns = [str(c).strip() for c in hdr]
    return out.dropna(how='all').dropna(axis=1, how='all')

def plausible(df):
    """Usable = we can find depths in it AND it holds elements or grain size.
    The depth test runs parse_depth rather than looking at column NAMES, because
    a Malvern export labels its samples 'GUAC_29A 15.5-16.0cm' and never writes
    the word 'depth' anywhere."""
    if df is None or not len(df): return None
    if _match(df.columns, EV_ALIASES_LIGHT) is not None: return 'events'
    k = classify(df)
    if k not in ('xrf','gs'): return None
    try: parse_depth(df, quiet=True)
    except Exception: return None
    return k

def corner(p, nrows=10, ncols=6):
    try: sh = _sheets(p)
    except Exception as ex: print("   unreadable:", ex); return
    for name, raw in sh.items():
        print(f"   --- sheet '{name}'  {raw.shape[0]} x {raw.shape[1]}")
        print(raw.iloc[:nrows, :ncols].to_string())

def smart_read(p, verbose=True):
    """Plain read first. If that is unusable, try EVERY sheet x header row
    (first 15) x orientation until one yields something with depths in it.
    Exports differ enough between instruments and operators that guessing a
    single layout does not hold up."""
    df = None
    try:
        df = read_any(p)
        if plausible(df): return df
    except Exception:
        pass
    for name, raw in _sheets(p).items():
        raw = raw.dropna(how='all').dropna(axis=1, how='all')
        if not len(raw): continue
        for tr in (False, True):
            nrows = raw.shape[1] if tr else raw.shape[0]
            for hr in range(min(15, nrows)):
                try: cand = _assemble(raw, hr, tr)
                except Exception: continue
                if not len(cand): continue
                if plausible(cand):
                    if verbose:
                        print(f"      {os.path.basename(p)}: sheet '{name}', "
                              f"header row {hr+1}"
                              f"{', transposed' if tr else ''} -> "
                              f"{cand.shape[0]} rows x {cand.shape[1]} cols")
                    return cand
    if verbose:
        print(f"   !! {os.path.basename(p)}: no readable layout found. "
              f"Top-left corner of each sheet:")
        corner(p)
    return df

def _norm(x):
    return re.sub(r'[^a-z0-9]', '', str(x).lower())

OWN_OUTPUTS = ('gs_stats_', 'event_table_', 'qc_report')   # files this notebook writes

def _own(path):
    b = os.path.basename(path).lower()
    return b.startswith(OWN_OUTPUTS) or b.endswith('_pca_scores.csv')

def data_files():
    return [f for f in sorted(glob.glob('/content/*'))
            if not os.path.isdir(f) and f.lower().endswith(DATA_EXT) and not _own(f)]

def _stem(path):
    """File name without extension and without Colab's ' (1)' re-upload suffix."""
    stem = os.path.splitext(os.path.basename(path))[0]
    return _norm(re.sub(r'\(\d+\)\s*$', '', stem).strip())

def resolve(spec):
    """Turn 'GUAC 29A GSD' into the real path, whatever its extension.
    A file matches when EVERY word of spec appears in its name (case, spaces,
    '-' and '_' ignored). Uploading the same file twice leaves 'name.xlsx' AND
    'name (1).xlsx' in /content; both match, so the NEWEST is used.
    No match -> a warning and None, and the notebook falls back to looking at
    the columns of every uploaded file for that core."""
    if not spec: return None
    words = [_norm(w) for w in re.split(r'[\s_\-]+', spec) if _norm(w)]
    hits = [f for f in data_files() if all(w in _stem(f) for w in words)]
    if not hits:
        print(f"   ! no file name contains all of {words} - will pick by content. "
              f"In /content: {[os.path.basename(f) for f in data_files()]}")
        return None
    hits.sort(key=os.path.getmtime, reverse=True)
    if len(hits) > 1:
        print(f"   ! {len(hits)} files match '{spec}' - using the "
              f"newest: {os.path.basename(hits[0])}")
        print(f"     ignored: {[os.path.basename(f) for f in hits[1:]]}"
              f"   (run CELL 0 to clear duplicates, or make FILES more specific)")
    return hits[0]

def candidates(tag):
    """Every data file whose name contains the core tag, ignoring - vs _ ."""
    t = _norm(tag)
    return [f for f in data_files() if t in _norm(os.path.basename(f))]

# ---- depth parsing ----------------------------------------------------------
# Handles all three layouts seen so far:
#   a) one numeric column        13.0, 13.2, 13.4      (24A, 2 mm)
#   b) interval strings          '15.5-16.0', '16.0-16.5cm'   (29A, 5 mm)
#   c) separate top/bottom columns
_IVAL = re.compile(r'(\d+\.?\d*)\s*(?:-|–|—|to)\s*(\d+\.?\d*)')

def _interval(text):
    """Get a depth out of a sample label, in order of preference:
         'GUAC_29A 15.5-16.0cm'  -> (15.5, 16.0)   interval
         'GUAC-24A-1G-1-W 13.0cm'-> (13.0, 13.0)   number tagged cm
         '... 13.0'              -> (13.0, 13.0)   last decimal number
    A bare integer is never taken as a depth, so the 24 in a core name and the
    1 in '1G-1-W' cannot be mistaken for one."""
    s = str(text)
    best = None
    for a, b in _IVAL.findall(s):
        a, b = float(a), float(b)
        if 0 <= a < b <= 500: best = (a, b)
    if best: return best
    m = re.findall(r'(\d+\.?\d*)\s*cm\b', s, re.I)
    if m and 0 <= float(m[-1]) <= 500: return (float(m[-1]), float(m[-1]))
    m = re.findall(r'\d+\.\d+', s)
    if m and 0 <= float(m[-1]) <= 500: return (float(m[-1]), float(m[-1]))
    return (np.nan, np.nan)

def _regular(v):
    """Depth series are evenly spaced; log-spaced size bins are not."""
    d = np.diff(np.sort(v.dropna().to_numpy(float)))
    d = d[d > 0]
    return len(d) >= 5 and d.std() / d.mean() < 0.3

def parse_depth(df, quiet=False):
    """Find depths, in this order:
       A. an explicit top+bottom pair, or a column named like a depth holding
          plain numbers;
       B. ANY column whose text carries a depth - '15.5-16.0cm', '13.0cm';
       C. last resort, an evenly spaced numeric column.
    B runs before C on purpose. A grain-size sheet has a column of size bins
    that is numeric and rising, and taking that as depth is exactly how a
    sheet gets read upside down."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    cn = {c: _norm(c) for c in df.columns}
    named = [c for c, n in cn.items()
             if 'depth' in n or 'interval' in n or 'sample' in n]
    top = next((c for c, n in cn.items()
                if n.startswith(('depthtop','top','upper','from'))), None)
    bot = next((c for c, n in cn.items()
                if n.startswith(('depthbot','depthbase','bottom','base','lower'))
                or n == 'to'), None)
    t = b = None; how = ''
    if top and bot:
        t = pd.to_numeric(df[top], errors='coerce')
        b = pd.to_numeric(df[bot], errors='coerce')
        how = f"columns '{top}' / '{bot}'"
    if t is None:                                            # A
        for c in named:
            num = pd.to_numeric(df[c], errors='coerce')
            if num.notna().mean() > 0.9:
                t = b = num; how = f"column '{c}'"; break
    if t is None:                                            # B
        for c in df.columns:
            probe = df[c].head(30)
            if pd.to_numeric(probe, errors='coerce').notna().mean() > 0.5:
                continue      # a plain number column is NOT a sample label;
                              # size bins like 0.375 would otherwise be read
                              # as depths of 0.375 cm
            ext = pd.DataFrame([_interval(v) for v in probe])
            if ext[0].notna().mean() > 0.5:
                full = pd.DataFrame([_interval(v) for v in df[c]], index=df.index)
                t, b = full[0], full[1]; how = f"labels in column '{c}'"; break
    if t is None:                                            # C
        for c in df.columns:
            num = pd.to_numeric(df[c], errors='coerce')
            if (num.notna().mean() > 0.9 and num.dropna().between(0, 500).all()
                    and _regular(num)):
                t = b = num; how = f"evenly spaced column '{c}'"; break
    assert t is not None, (f"no depth / interval column found in "
                           f"{list(df.columns)[:8]} ... run CELL 1b")
    if not quiet: print(f"        depths read from {how}")
    df['depth_top'] = np.minimum(t, b)
    df['depth_bot'] = np.maximum(t, b)
    df['depth_cm']  = (df['depth_top'] + df['depth_bot']) / 2
    return df.dropna(subset=['depth_cm'])

def resolution(v):
    d = np.diff(np.sort(np.unique(np.asarray(v, float))))
    return float(np.median(d)) if len(d) else float('nan')

# ---- grain size: percentile columns, or compute them from raw bins ----------
PCT_KEYS = {'Dmean_um': ('mean','dmean','meansize','graphicmean','mz','fwmean',
                         'folkwardmean','folkandwardmean'),
            'D10_um': ('d10','dx10','dv10','dv010'),
            'D50_um': ('d50','dx50','dv50','dv050','median'),
            'D90_um': ('d90','dx90','dv90','dv090'),
            'sorting_phi': ('sorting','sortingphi','sd')}

def pct_cols(df):
    out = {}
    for c in df.columns:
        n = _norm(c)
        for key, pats in PCT_KEYS.items():
            if key not in out and any(n == p or n.startswith(p) for p in pats):
                out[key] = c
    return out

def _size(c):
    """'0.375', '1.02 um', '2000µm' -> float µm.  NOTE: parse the RAW header,
    never the normalised one - _norm() strips the decimal point and would turn
    0.375 µm into 375 µm."""
    t = re.sub(r'\s*(um|µm|μm|microns?)\s*$', '', str(c).strip(), flags=re.I)
    try:
        v = float(t)
    except ValueError:
        return None
    return v if v > 0 else None

def bin_cols(df):
    """Columns whose header is a grain size. Headers in phi (they never go
    above ~20, while µm bins always reach hundreds) are converted to µm."""
    b = [(c, _size_any(c)) for c in df.columns
         if str(c) not in ('depth_cm','depth_top','depth_bot')]
    b = [(c, v) for c, v in b if v is not None]
    if len(b) < 10: return []
    if max(v for _, v in b) < 30:                       # phi headers
        b = [(c, float(phi_to_um(v))) for c, v in b]
    else:
        b = [(c, v) for c, v in b if v > 0]             # µm: no zero-size bin
    return b if len(b) >= 10 else []

def _size_any(c):
    """Like _size, but also accepts 0 and negative numbers (phi headers)."""
    t = re.sub(r'\s*(um|µm|μm|microns?|phi|φ)\s*$', '', str(c).strip(), flags=re.I)
    try:
        return float(t)
    except ValueError:
        return None

# ---- units: phi or µm? --------------------------------------------------------
# Folk & Ward work on the PHI scale:  phi = -log2(size in mm).
#   big phi = FINE grain, small phi = COARSE grain.  4 phi = 62.5 µm, 8 phi = 3.9 µm
# Programs like GRADISTAT give the Folk & Ward mean in phi (e.g. 5.2), while the
# figures here always plot µm. So every size column is checked and converted.
phi_to_um = lambda phi: 1000.0 * 2.0 ** (-np.asarray(phi, float))

def size_unit(colname, values, d50_um=None):
    """'phi' or 'um' for a grain-SIZE column (mean, D10, D50, D90).
    1. the column name says it:  'Mean (phi)', 'MEAN φ', 'Mz phi'  /  'Mean (µm)'
    2. otherwise, if a D50 in µm is known, read the column both ways and keep
       the reading closer to D50 (mean and median are always near each other)
    3. otherwise: values all between -2 and 12 are taken as phi."""
    n = str(colname).lower()
    if 'phi' in n or 'φ' in n or '(f)' in n: return 'phi'
    if any(u in n for u in ('µm', 'μm', 'um', 'micron')): return 'um'
    v = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if not len(v): return 'um'
    if d50_um is not None and np.isfinite(d50_um) and d50_um > 0:
        # read it both ways; the reading that lands closer to D50 is right
        as_um  = abs(np.log(max(v.median(), 1e-6) / d50_um))
        as_phi = abs(np.log(phi_to_um(v.median()) / d50_um))
        return 'um' if as_um < as_phi else 'phi'
    return 'phi' if (v.between(-2, 12).all() and v.median() < 10) else 'um'

def from_bins(df, bins):
    """All grain-size statistics from the raw volume-% size bins.

    Steps, in plain words:
      1. sort the bins from fine to coarse
      2. add them up -> cumulative % finer than each size (0 ... 100 %)
      3. read off the size at 5, 10, 16, 50, 84, 90, 95 % by straight-line
         interpolation on a log-size axis (grain size is log-normal-ish)
      4. Folk & Ward (1957) graphic mean, sorting, skewness, kurtosis
      5. % clay / silt / sand from the same cumulative curve"""
    sz = np.array([v for _, v in bins]); o = np.argsort(sz); sz = sz[o]
    V = df[[c for c, _ in bins]].apply(pd.to_numeric, errors='coerce') \
          .to_numpy(float)[:, o]
    V = np.where(np.isfinite(V), V, 0.0)
    # the cumulative value belongs at the UPPER edge of each size class
    r = np.exp(np.mean(np.diff(np.log(sz))))          # bin width ratio
    if   BIN_EDGE == 'lower': top = np.append(sz[1:], sz[-1] * r)
    elif BIN_EDGE == 'mid':   top = sz * np.sqrt(r)
    else:                     top = sz
    edges = np.concatenate([[top[0] / r], top])         # add the finest edge
    logd = np.log10(edges)
    cum = np.cumsum(V, axis=1); tot = cum[:, -1]
    cumpct = np.hstack([np.zeros((len(V), 1)),
                        cum / np.where(tot > 0, tot, np.nan)[:, None] * 100])

    def size_at(p):                     # size (µm) at p % finer
        out = np.full(len(V), np.nan)
        for i in range(len(V)):
            if tot[i] > 0: out[i] = 10 ** np.interp(p, cumpct[i], logd)
        return out
    def pct_finer(d_um):                # % finer than a size
        out = np.full(len(V), np.nan)
        for i in range(len(V)):
            if tot[i] > 0: out[i] = np.interp(np.log10(d_um), logd, cumpct[i])
        return out

    phi = lambda d: -np.log2(np.asarray(d) / 1000.0)   # µm -> phi
    # Folk & Ward count percentiles from the COARSE end on the phi scale:
    # phi84 is the size with 84 % coarser = 16 % finer. Hence 100 - p.
    P = {p: phi(size_at(100 - p)) for p in (5, 16, 25, 50, 75, 84, 95)}
    res = pd.DataFrame(index=df.index)
    res['D10_um'] = size_at(10); res['D50_um'] = size_at(50); res['D90_um'] = size_at(90)
    Mz = (P[16] + P[50] + P[84]) / 3
    res['Dmean_um']    = 2.0 ** (-Mz) * 1000                        # Folk & Ward mean
    res['sorting_phi'] = (P[84] - P[16]) / 4 + (P[95] - P[5]) / 6.6
    res['skewness']    = ((P[16] + P[84] - 2 * P[50]) / (2 * (P[84] - P[16])) +
                          (P[5]  + P[95] - 2 * P[50]) / (2 * (P[95] - P[5])))
    res['kurtosis']    = (P[95] - P[5]) / (2.44 * (P[75] - P[25]))
    res['Mz_phi'] = Mz                                  # Folk & Ward mean in phi
    clay, sand = CLAY_SILT_SAND
    res['clay_pct'] = pct_finer(clay)
    res['silt_pct'] = pct_finer(sand) - res['clay_pct']
    res['sand_pct'] = 100 - pct_finer(sand)
    # % of the sample in every Udden-Wentworth class (they add up to ~100)
    for name, lo, hi in WENTWORTH:
        below_hi = pct_finer(hi)
        below_lo = pct_finer(lo) if lo > 0 else 0.0
        res[f"pct_{name.replace(' ', '_')}"] = below_hi - below_lo
    return res

# ---- words for the numbers: Folk & Ward (1957) and Udden-Wentworth ---------
# Each list is (upper limit, name), read from the top: the first limit that the
# value is below gives the name.
FW_SORTING = [(0.35, 'very well sorted'), (0.50, 'well sorted'),
              (0.71, 'moderately well sorted'), (1.00, 'moderately sorted'),
              (2.00, 'poorly sorted'), (4.00, 'very poorly sorted'),
              (np.inf, 'extremely poorly sorted')]
FW_SKEWNESS = [(-0.30, 'strongly coarse skewed'), (-0.10, 'coarse skewed'),
               (0.10, 'near symmetrical'), (0.30, 'fine skewed'),
               (np.inf, 'strongly fine skewed')]
FW_KURTOSIS = [(0.67, 'very platykurtic'), (0.90, 'platykurtic'),
               (1.11, 'mesokurtic'), (1.50, 'leptokurtic'),
               (3.00, 'very leptokurtic'), (np.inf, 'extremely leptokurtic')]

def _word(v, table):
    if not np.isfinite(v): return ''
    return next(name for limit, name in table if v < limit)

def wentworth_class(d_um):
    """Udden-Wentworth class name of a size in µm, e.g. 21 -> 'medium silt'."""
    if not np.isfinite(d_um): return ''
    for name, lo, hi in WENTWORTH:
        if lo <= d_um < hi: return name
    return 'coarser than granule'

def describe_gs(out):
    """Add the class name of the mean, and the Folk & Ward words for sorting,
    skewness and kurtosis, as text columns next to the numbers."""
    if 'Dmean_um' in out:
        out['mean_class']     = [wentworth_class(v) for v in out['Dmean_um']]
    if 'sorting_phi' in out:
        out['sorting_class']  = [_word(v, FW_SORTING)  for v in out['sorting_phi']]
    if 'skewness' in out:
        out['skewness_class'] = [_word(v, FW_SKEWNESS) for v in out['skewness']]
    if 'kurtosis' in out:
        out['kurtosis_class'] = [_word(v, FW_KURTOSIS) for v in out['kurtosis']]
    return out

# ---- clr, Bertrand et al. (2024) Eqs. 1-2 -----------------------------------
def clr(frame, cols, suffix='_clr'):
    """ln(x) - mean(ln(x)) row-wise == ln(x / geometric mean). The log form is
    used because the PRODUCT of 10 values ~1e5 each overflows."""
    X = frame[cols].to_numpy(float)
    assert (X > 0).all(), "clr requires strictly positive counts"
    L = np.log(X)
    out = pd.DataFrame(L - L.mean(axis=1, keepdims=True),
                       columns=[f'{c}{suffix}' for c in cols], index=frame.index)
    assert np.abs(out.sum(axis=1)).max() < 1e-10, "clr checksum failed"
    return out

_e = ['Si','Cl','K','Ca','Ti','Mn','Fe','Zn','Br','Rb','Sr','Zr','Ba','Pb']
_c = pd.DataFrame([[2738,4416,26753,27445,19923,23908,951891,5278,4322,
                    14393,14990,2601,1077,18565]], columns=_e)
_pub = np.array([-1.462,-0.984,0.817,0.843,0.523,0.705,4.389,-0.806,
                 -1.005,0.198,0.238,-1.513,-2.395,0.452])
_dev = np.abs(clr(_c,_e).to_numpy()[0] - _pub).max()
assert _dev < 0.002, f"clr does NOT match published values (dev {_dev})"
print(f"\nclr unit test PASS - max deviation {_dev:.5f} vs Bertrand Appendix 1")

# ---- decide what each file IS, from its columns, not its name --------------
NEED = sorted(set(ELS) | {a for a,b,_,_ in RATIOS} | {b for a,b,_,_ in RATIOS})

def classify(df):
    cols = {str(c).strip() for c in df.columns}
    if sum(e in cols for e in NEED) >= max(3, len(NEED)//2): return 'xrf'
    if pct_cols(df) or bin_cols(df): return 'gs'
    return '?'

def pick_files(tag, core):
    """Named files win; a slot left as None falls back to content detection."""
    want = FILES.get(core, {})
    chosen, cache = {}, {}
    for k in ('xrf','gs'):
        p = resolve(want.get(k))
        if p:
            df = smart_read(p); kind = classify(df); cache[p] = (kind, df)
            if kind != k:
                print(f"   ! {os.path.basename(p)} was named as {k.upper()} but "
                      f"its columns look like '{kind}' - using it as {k.upper()} anyway")
            chosen[k] = p
    for p in candidates(tag):
        if p in cache: continue
        try: df = smart_read(p)
        except Exception as ex: print(f"   skip {os.path.basename(p)}: {ex}"); continue
        kind = classify(df); cache[p] = (kind, df)
        if kind in ('xrf','gs') and not chosen.get(kind): chosen[kind] = p
    for k in ('xrf','gs'):
        p = chosen.get(k)
        if p and p not in cache: cache[p] = (k, smart_read(p))
    print(f"\n{core}")
    for p, (kind, _) in cache.items():
        star = '<-' if p in (chosen.get('xrf'), chosen.get('gs')) else '  '
        print(f"   {star} {kind:4s} {os.path.basename(p)}")
    return chosen, cache

# ---- load ------------------------------------------------------------------
def load_core(core, path, df):
    r = parse_depth(df)
    r.columns = [str(c).strip() for c in r.columns]
    miss = [e for e in NEED if e not in r.columns]
    assert not miss, f"{core}: XRF file missing {miss}. Found {list(r.columns)}"
    r = r[['depth_cm']+NEED].apply(pd.to_numeric, errors='coerce').dropna()
    r = r.groupby('depth_cm', as_index=False).mean().sort_values('depth_cm')
    r = r.reset_index(drop=True)
    for e in NEED:                        # zeros -> half the minimum (Sec 2.2.2b)
        nz = r.loc[r[e] > 0, e]; n0 = int((r[e] <= 0).sum())
        if n0:
            r.loc[r[e] <= 0, e] = nz.min()/2
            print(f"   {core} {e}: {n0} zeros -> {nz.min()/2:.1f}")
    d = pd.concat([r, clr(r, ELS)], axis=1)
    for a, b, lab, _ in RATIOS:
        d[f'ln_{a}_{b}'] = np.log(d[a] / d[b])            # from RAW counts
    print(f"   XRF: {len(d)} rows, {d.depth_cm.min():.2f}-{d.depth_cm.max():.2f} cm, "
          f"step {resolution(d.depth_cm)*10:.0f} mm, clr + {len(RATIOS)} log-ratios")
    return d

GS_KEYS = ['D10_um','D50_um','D90_um','Dmean_um','sorting_phi']

def load_gs(core, path, df):
    """Grain size for one core. A raw Malvern master file usually holds
    SEVERAL runs of each sample (replicates). They are averaged bin by bin
    first, and the statistics are then calculated once per sample."""
    if path is None: print(f"   GS : none found"); return None
    g = parse_depth(df)
    pcs, bins = pct_cols(g), bin_cols(g)
    use_bins = bool(bins) and (GS_SOURCE == 'bins' or
                               (GS_SOURCE == 'auto') or not pcs)
    if GS_SOURCE == 'columns' and pcs: use_bins = False
    if use_bins:
        B = g[[c for c, _ in bins]].apply(pd.to_numeric, errors='coerce')
        B['depth_cm'] = g['depth_cm'].values
        B = B[B.drop(columns='depth_cm').fillna(0).sum(axis=1) > 0]   # empty rows out
        runs = B.groupby('depth_cm').size()
        B = B.groupby('depth_cm', as_index=False).mean()             # average runs
        out = from_bins(B, bins)
        out.insert(0, 'depth_cm', B['depth_cm'].values)
        how = (f"{len(bins)} raw size bins "
               f"({min(v for _,v in bins):.2f}-{max(v for _,v in bins):.0f} µm, "
               f"BIN_EDGE='{BIN_EDGE}') -> all statistics calculated here; "
               f"{runs.mean():.1f} runs averaged per sample "
               f"(min {runs.min()}, max {runs.max()})")
        if pcs.get('D50_um'):          # the file has Malvern's own D50: compare
            m = pd.DataFrame({'depth_cm': g['depth_cm'].values,
                              'mal': pd.to_numeric(g[pcs['D50_um']], errors='coerce')})
            m = m.groupby('depth_cm', as_index=False).mean().merge(out[['depth_cm','D50_um']])
            dev = (m.D50_um / m.mal - 1).abs().median() * 100
            print(f"        check vs Malvern D50: median difference {dev:.1f}%"
                  f"{'   <- OK' if dev < 3 else '   <- try another BIN_EDGE'}")
    elif pcs:
        out = pd.DataFrame({k: pd.to_numeric(g[c], errors='coerce')
                            for k, c in pcs.items()})
        # every SIZE column -> µm (sorting stays in phi, as Folk & Ward define it)
        d50 = None
        if 'D50_um' in pcs and size_unit(pcs['D50_um'], out['D50_um']) == 'um':
            d50 = out['D50_um'].median()
        units = {}
        for k in ('Dmean_um', 'D10_um', 'D50_um', 'D90_um'):
            if k not in out: continue
            u = size_unit(pcs[k], out[k], None if k == 'D50_um' else d50)
            units[pcs[k]] = u
            if u == 'phi':
                out[f"{k.replace('_um', '')}_phi"] = out[k]      # keep the phi value
                out[k] = phi_to_um(out[k])                       # plot in µm
        if 'sorting_phi' in pcs:
            sn = str(pcs['sorting_phi']).lower()
            if any(u in sn for u in ('µm', 'μm', 'um', 'micron')):
                # geometric sorting (a ratio, >= 1) -> phi: sorting_phi = log2(value)
                out['sorting_phi'] = np.log2(out['sorting_phi'])
                print(f"        sorting '{pcs['sorting_phi']}' is geometric (µm) "
                      f"-> converted to phi with log2")
        conv = [c for c, u in units.items() if u == 'phi']
        if conv:
            print(f"        phi -> µm converted: {conv}  (µm = 1000 x 2^-phi)")
        print(f"        units detected: {units}   <-- CHECK THESE")
        out.insert(0, 'depth_cm', g['depth_cm'].values)
        how = f"percentile columns {list(pcs.values())}"
    else:
        raise AssertionError(f"{core} grain size: no D10/D50/D90 columns and no "
                             f"size bins recognised in {list(g.columns)[:12]}... "
                             f"run CELL 1b to see the file layout")
    if 'Dmean_um' not in out:      # percentile-only file: Folk mean on D10/50/90
        phi = lambda d: -np.log2(pd.to_numeric(d, errors='coerce') / 1000.0)
        if {'D10_um','D50_um','D90_um'} <= set(out.columns):
            out['Dmean_um'] = 2.0 ** (-(phi(out.D10_um) + phi(out.D50_um)
                                        + phi(out.D90_um)) / 3) * 1000
            print("        no mean column - approximated from D10/D50/D90 "
                  "(NOT the Folk & Ward formula - upload raw bins for that)")
    if 'sorting_phi' not in out and {'D10_um','D90_um'} <= set(out.columns):
        phi = lambda d: -np.log2(pd.to_numeric(d, errors='coerce') / 1000.0)
        out['sorting_phi'] = (phi(out.D10_um) - phi(out.D90_um)) / 2.56
        print("        no sorting column - approximated from D10/D90 "
              "(NOT the Folk & Ward formula - upload raw bins for that)")
    out = out.dropna(subset=[k for k in GS_KEYS if k in out], how='all')
    out = out.groupby('depth_cm', as_index=False).mean().sort_values('depth_cm')
    out = describe_gs(out.reset_index(drop=True))
    gaps = {k: int(out[k].isna().sum()) for k in GS_KEYS if k in out}
    print(f"   GS : {len(out)} depths, {out.depth_cm.min():.2f}-"
          f"{out.depth_cm.max():.2f} cm, step {resolution(out.depth_cm)*10:.0f} mm")
    print(f"        {how}")
    if any(gaps.values()):
        print(f"        missing values kept as gaps: {gaps}")
    print(f"        first depths parsed: "
          f"{[round(v,2) for v in out.depth_cm.head(4)]} ... "
          f"{[round(v,2) for v in out.depth_cm.tail(2)]}   <-- CHECK THESE")
    if 'mean_class' in out:
        print(f"        mean size classes (Udden-Wentworth): "
              f"{out.mean_class.value_counts().to_dict()}")
    if 'sorting_class' in out and 'skewness' in out:
        print(f"        sorting (Folk & Ward 1957): "
              f"{out.sorting_class.value_counts().to_dict()}")
    fn = f"/content/GS_STATS_{core}.csv"
    out.to_csv(fn, index=False)
    print(f"        statistics table -> {fn}  (compare a few rows with the Malvern report)")
    return out

def load_pair(tag, core):
    chosen, cache = pick_files(tag, core)
    assert chosen.get('xrf'), (f"{core}: no XRF file found. Uploaded names must "
                               f"contain '{tag}'.")
    px = chosen['xrf']; pg = chosen.get('gs')
    return (load_core(core, px, cache[px][1]),
            load_gs(core, pg, cache[pg][1] if pg else None))

_29 = load_pair('29A', 'GUAC-29A')
_24 = load_pair('24A', 'GUAC-24A')
CORES = {'GUAC-29A': _29[0], 'GUAC-24A': _24[0]}
GS    = {'GUAC-29A': _29[1], 'GUAC-24A': _24[1]}
print("\nDone. Run Cell 2.")

## Cell 1b – inspector (run only if a file does not load)

In [ ]:
# =============================================================================
# CELL 1b - INSPECTOR. Run this only if a file fails to load. It prints the
#           top-left corner of every sheet so the real layout is visible.
# =============================================================================
LOOK_AT = 'GSD'          # substring of the file name; '' = every uploaded file
NROWS, NCOLS = 12, 8

for p in sorted(glob.glob('/content/*')):
    if os.path.isdir(p) or not p.lower().endswith(DATA_EXT): continue
    if LOOK_AT and LOOK_AT.lower() not in os.path.basename(p).lower(): continue
    print("=" * 78); print(os.path.basename(p))
    try:
        sh = _sheets(p)
    except Exception as ex:
        print("   unreadable:", ex); continue
    for name, raw in sh.items():
        print(f"\n--- sheet '{name}'   {raw.shape[0]} rows x {raw.shape[1]} cols")
        print(raw.iloc[:NROWS, :NCOLS].to_string())
        hr, tr = find_header(raw.dropna(how='all').dropna(axis=1, how='all'))
        print(f"   guessed header row {hr+1}"
              f"{', transposed (samples in columns)' if tr else ''}")

## Cell 2 – events, ages, colours and drawing helpers
Each event gets **one colour** that is used in *every* figure (profiles, PCA,
LDA, Fig. 12 …), and E07 is the same colour in 24A and 29A.

New here: `BREAK_AT_GAPS = True` makes a line stop where a grain-size sample is
missing, so deleted points are not hidden behind a straight line.

In [ ]:
# =============================================================================
# CELL 2 - AGES, EVENTS, QC FILTER, DEPTH WINDOW, COLOURS, HELPERS
#          <<< fill TYPES_29A / TYPES_24A with 'A' / 'B' when classified >>>
# =============================================================================

# ---- age labels -------------------------------------------------------------
AGE_DEC  = 1        # decimals on age AND uncertainty (0 = whole years)
AGE_SIDE = 'right'  # 'right' = just outside the panel; 'in' = inside
AGE_GAP  = 1.8      # cm: minimum vertical spacing between labels
AGE_FS   = 6.5      # label font size

# ---- where events and ages come from ---------------------------------------
EVENTS_FILE = None   # None  = use an uploaded event table if one is found,
                     #         otherwise fall back to the built-in lists below.
                     # 'x.csv' = force that file.  False = always use built-ins.
# Recognised columns (any spelling/case, extra columns ignored):
#   core | event | top_cm | base_cm | age_CE | age_err | type
# 'core' is optional if the file name carries 24A / 29A.

# ---- event QC ---------------------------------------------------------------
MIN_SAMPLES = 4     # an event needs >= this many samples to be plotted
MIN_SAMPLES_BASIS = 'either'
# 'xrf'    judge on XRF only
# 'gs'     judge on grain size only
# 'either' keep if EITHER dataset clears the bar   <- default
# 'both'   keep only if both do
# 'either' matters because the two run at different resolutions: a 1.0 cm event
# in 24A holds 6 grain-size samples at 2 mm but only 3 XRF samples at 5 mm.
# An event with no GSD at all (n_samples = 0 in the event table) is a very
# small event and is kept only if its XRF coverage stands on its own.
SNAP_TOL_CM = 0.2   # if an event boundary falls between samples, pull in the
                    # nearest sample just OUTSIDE it when the offset is <= this.
                    # 0.2 cm => a 1 mm or 2 mm shortfall is included, 3 mm is
                    # not. The borrowed sample is skipped if it already belongs
                    # to a neighbouring event.
CONTIG_TOL  = 0.15  # cm: events closer than this are drawn as touching

# ---- missing samples (e.g. the fake Malvern points you deleted) -------------
BREAK_AT_GAPS = True  # True  = the line stops where a sample is missing
                      # False = the line joins across the gap
GAP_FACTOR    = 1.6   # a "gap" = two samples further apart than 1.6 x the
                      # normal sample spacing of that dataset
BOUNDARY_KW = dict(color='0.35', lw=0.8, ls=(0,(4,3)), zorder=1.5)

# ---- plotted depth window (display only; PCA still uses every row) ----------
DEPTH_LIM = {'GUAC-29A': (15.0, 88.0),      # (top_cm, bottom_cm)
             'GUAC-24A': (12.0, 88.0)}

# ---- colour scheme ----------------------------------------------------------
COLOR_BY = 'event'  # 'event'  one colour per event (best for classifying)
                    # 'type'   blue = Type A, red = Type B (publication)
                    # 'element' Bertrand style, one colour per variable

# ---- core photo panel -------------------------------------------------------
PHOTO_MODE   = 'space'   # 'space' blank depth-registered box for PowerPoint
                         # 'image' draw the photo from PHOTO_CFG
                         # 'none'  no photo column at all
PHOTO_W      = 0.55      # width of the photo column (data panels = 1.0)
PHOTO_GUIDE  = True      # dashed outline + caption in 'space' mode;
                         # set False for a truly empty box in the final export
PHOTO_CFG = {            # only used when PHOTO_MODE = 'image'
    'GUAC-29A': dict(path='', top_cm=0.0, bot_cm=100.0, crop_lr=(0.0, 1.0)),
    'GUAC-24A': dict(path='', top_cm=0.0, bot_cm=100.0, crop_lr=(0.0, 1.0))}

# =============================================================================
DEF_AGES = {1:(2010.5,1.7), 2:(2000.1,1.4), 3:(1977.9,2.5), 4:(1966.1,3.4),
        5:(1962.0,3.7), 6:(1960.6,3.7), 7:(1960.6,3.7), 8:(1931.5,2.3),
        9:(1921.0,3.9),10:(1902.1,3.7),11:(1895.6,3.9),12:(1893.8,4.3),
       13:(1886.7,4.8),14:(1881.3,5.3),15:(1861.9,6.2),16:(1859.9,6.5),
       17:(1845.3,6.6),18:(1836.7,3.8),19:(1820.6,4.4),20:(1795.5,3.5)}

DEF_EVENTS_29A = {1:(13.40,14.10), 2:(15.10,19.10), 3:(22.60,30.15), 4:(31.30,33.20),
              5:(33.55,36.80), 6:(36.80,41.00), 7:(41.00,49.10), 8:(53.25,55.80),
              9:(56.80,58.10),10:(60.20,60.60),11:(61.20,64.10),12:(64.65,66.60),
             13:(67.15,67.50),14:(67.90,68.30),15:(70.80,71.15),16:(71.25,71.50),
             17:(73.15,75.40),18:(76.65,80.00),19:(87.75,87.75)}
DEF_EVENTS_24A = {1:(11.75,11.90), 2:(12.90,17.40), 3:(19.35,26.20), 4:(27.50,27.85),
              5:(28.35,30.80), 6:(30.90,33.00), 7:(33.00,48.00), 8:(51.70,53.00),
             10:(55.70,55.85),11:(56.50,62.00),14:(63.50,63.75),15:(66.25,66.50),
             16:(66.70,67.10),17:(68.80,69.75),18:(70.15,70.55),19:(71.30,72.70),
             20:(75.20,79.80)}

DEF_TYPES_29A = {e:'?' for e in DEF_EVENTS_29A}   # <-- 'A' / 'B' from grain size
DEF_TYPES_24A = {e:'?' for e in DEF_EVENTS_24A}

TYPE_COLOR = {'A':'#3182bd','B':'#de2d26','?':'#969696','bg':'#cccccc'}
TYPE_LABEL = {'A':'Type A (flood/hurricane)','B':'Type B (earthquake)',
              '?':'Unclassified','bg':'background'}
ELCOL = {'Si':'#4299e1','K':'#dd6b20','Ca':'#2b6cb0','Ti':'#2c7a7b','Mn':'#38a169',
         'Fe':'#c05621','Sr':'#805ad5','Zr':'#b83280','S':'#975a16','Al':'#5a67d8'}
GSCOL = {'D10_um':'#a0aec0','D50_um':'#4a5568','D90_um':'#1a202c','sorting_phi':'#805ad5'}
GSMARK = {'D10_um':'o','D50_um':'s','D90_um':'^','sorting_phi':'D'}
PALETTE = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd','#8c564b',
           '#e377c2','#17becf','#bcbd22','#7f3f00','#3f51b5','#00897b']

# ---- optional: read events / ages / types from an uploaded table ------------
EV_ALIASES = {
 'event': ('event','eventid','eid','eventno','eventnumber','eventnum','id','no'),
 'core' : ('core','corename','coreid','corelabel'),
 'top'  : ('top','topcm','depthtop','eventtop','upper','from','start'),
 'base' : ('base','basecm','bottom','bottomcm','depthbase','depthbot','eventbase',
           'lower','to','end'),
 'age'  : ('age','agece','ageyr','ageyear','year','medianage','ce','bestage'),
 'err'  : ('err','error','ageerr','ageerror','ageunc','unc','uncertainty',
           'ageuncertainty','sigma','1sigma','sd','pm','plusminus','range'),
 'type' : ('type','eventtype','class','classification','category'),
}

def _match(cols, keys):
    for c in cols:
        n = _norm(c)
        if any(n == k for k in keys): return c
    for c in cols:
        n = _norm(c)
        if any(n.startswith(k) for k in keys): return c
    return None

def _evnum(v):
    m = re.search(r'(\d+)', str(v))
    return int(m.group(1)) if m else None

CORE_TAGS = (('24a', 'GUAC-24A'), ('29a', 'GUAC-29A'))

def wide_blocks(df):
    """Spot a table with one COLUMN BLOCK per core:
       age_CE | age_unc | event | in_24A | 24A_top_cm | 24A_base_cm |
               24A_n_samples | event | in_29A | 29A_top_cm | ...
    Returns {core: {'top':col, 'base':col, 'n':col, 'in':col}}."""
    blocks = {}
    for c in df.columns:
        n = _norm(c)
        for tag, core in CORE_TAGS:
            if not (n.startswith(tag) or n.startswith('in' + tag)): continue
            b = blocks.setdefault(core, {})
            if   n.endswith(('topcm','top')):                 b['top']  = c
            elif n.endswith(('basecm','base','bottomcm')):     b['base'] = c
            elif 'nsample' in n:                               b['n']    = c
            elif n.startswith('in'):                           b['in']   = c
    return {k: v for k, v in blocks.items() if 'top' in v and 'base' in v}

def load_event_table_wide(df, blocks):
    ecol = next((c for c in df.columns if _norm(c) == 'event'), None)
    acol = _match(df.columns, EV_ALIASES['age'])
    ucol = _match(df.columns, EV_ALIASES['err'])
    ev  = {c: {} for c, _ in [(v, 0) for v in blocks]} or {}
    ev  = {c: {} for c in blocks}
    typ = {c: {} for c in blocks}
    ages= {c: {} for c in blocks}
    nfile = {c: {} for c in blocks}
    absent = {c: [] for c in blocks}
    for _, r in df.iterrows():
        num = _evnum(r[ecol]) if ecol else None
        if num is None: continue
        for core, b in blocks.items():
            flag = str(r[b['in']]).strip().lower() if 'in' in b else 'yes'
            t = pd.to_numeric(r[b['top']],  errors='coerce')
            u = pd.to_numeric(r[b['base']], errors='coerce')
            if flag.startswith(('absent','no','n/a')) or not (
                    np.isfinite(t) and np.isfinite(u)):
                absent[core].append(f'E{num:02d}'); continue
            ev[core][num]  = (float(min(t, u)), float(max(t, u)))
            typ[core][num] = '?'
            if acol is not None:
                a = pd.to_numeric(r[acol], errors='coerce')
                e = pd.to_numeric(r[ucol], errors='coerce') if ucol else np.nan
                if np.isfinite(a):
                    ages[core][num] = (float(a), float(e) if np.isfinite(e) else 0.0)
            if 'n' in b:
                v = pd.to_numeric(r[b['n']], errors='coerce')
                if np.isfinite(v): nfile[core][num] = int(v)
    print(f"Event table: wide layout, one block per core")
    for core in blocks:
        print(f"   {core}: {len(ev[core])} events"
              + (f", ABSENT in this core: {absent[core]}" if absent[core] else ""))
    return ev, typ, ages, nfile

def find_event_table():
    """Any uploaded data file that looks like an event list (has an event
    column plus a top and a base)."""
    if EVENTS_FILE is False: return None
    if EVENTS_FILE: return EVENTS_FILE
    cands = [p for p in glob.glob('/content/*')
             if not os.path.isdir(p) and p.lower().endswith(DATA_EXT)]
    cands.sort(key=os.path.getmtime, reverse=True)      # newest upload first
    for p in cands:
        if os.path.basename(p).startswith('EVENT_TABLE_'): continue   # our own export
        try: df = smart_read(p)
        except Exception: continue
        c = df.columns
        if wide_blocks(df): return p
        if _match(c, EV_ALIASES['event']) and _match(c, EV_ALIASES['top']) \
           and _match(c, EV_ALIASES['base']):
            return p
    return None

def load_event_table(path):
    df = smart_read(path); df.columns = [str(x).strip() for x in df.columns]
    blocks = wide_blocks(df)
    if blocks:
        return load_event_table_wide(df, blocks)
    col = {k: _match(df.columns, v) for k, v in EV_ALIASES.items()}
    ev   = {c: {} for c in ('GUAC-29A','GUAC-24A')}
    typ  = {c: {} for c in ev}
    ages = {c: {} for c in ev}
    fname = _norm(os.path.basename(path))
    for _, r in df.iterrows():
        n = _evnum(r[col['event']])
        if n is None: continue
        if col['core']:
            cn = _norm(r[col['core']])
            core = 'GUAC-29A' if '29a' in cn else ('GUAC-24A' if '24a' in cn else None)
        else:
            core = ('GUAC-29A' if '29a' in fname else
                    'GUAC-24A' if '24a' in fname else None)
        if core is None: continue
        t, b = pd.to_numeric(r[col['top']], errors='coerce'), \
               pd.to_numeric(r[col['base']], errors='coerce')
        if not (np.isfinite(t) and np.isfinite(b)): continue
        ev[core][n] = (float(min(t, b)), float(max(t, b)))
        tv = str(r[col['type']]).strip().upper()[:1] if col['type'] else ''
        typ[core][n] = tv if tv in ('A','B') else '?'
        if col['age']:
            a = pd.to_numeric(r[col['age']], errors='coerce')
            e = pd.to_numeric(r[col['err']], errors='coerce') if col['err'] else np.nan
            if np.isfinite(a): ages[core][n] = (float(a), float(e) if np.isfinite(e) else 0.0)
    used = {k: v for k, v in col.items() if v}
    print(f"Event table <- {os.path.basename(path)}   columns used: {used}")
    for c in ev: print(f"   {c}: {len(ev[c])} events"
                       f"{'' if ev[c] else '  (none - check the core column)'}")
    return ev, typ, ages, {c: {} for c in ev}

_tbl = find_event_table()
N_FROM_FILE = {'GUAC-29A': {}, 'GUAC-24A': {}}
if _tbl:
    print(f"Event table <- {os.path.basename(_tbl)}")
    _ev, _ty, _ag, _nf = load_event_table(_tbl)
    for c in N_FROM_FILE: N_FROM_FILE[c] = _nf.get(c, {})
    EVENTS_29A = _ev.get('GUAC-29A') or DEF_EVENTS_29A
    EVENTS_24A = _ev.get('GUAC-24A') or DEF_EVENTS_24A
    TYPES_29A  = {e: _ty.get('GUAC-29A', {}).get(e, '?') for e in EVENTS_29A}
    TYPES_24A  = {e: _ty.get('GUAC-24A', {}).get(e, '?') for e in EVENTS_24A}
    AGES_BY_CORE = {'GUAC-29A': _ag.get('GUAC-29A') or DEF_AGES,
                    'GUAC-24A': _ag.get('GUAC-24A') or DEF_AGES}
else:
    print("No event table uploaded - using the built-in event lists.")
    EVENTS_29A, EVENTS_24A = DEF_EVENTS_29A, DEF_EVENTS_24A
    TYPES_29A,  TYPES_24A  = DEF_TYPES_29A,  DEF_TYPES_24A
    AGES_BY_CORE = {'GUAC-29A': DEF_AGES, 'GUAC-24A': DEF_AGES}

# ---- build the event list, dropping the thin ones ---------------------------
def _occupied(core_events, dep, skip):
    return any(eid != skip and t <= dep <= b for eid, t, b, *_ in core_events)

def event_slice(df, core_events, eid, top, base, col=None, snap=None):
    """Rows of df belonging to one event, with boundary snapping.

    Samples strictly inside [top, base] always count. If the boundary sits
    between samples, the single nearest sample outside is added when it is
    within SNAP_TOL_CM and is not inside another event. `col` restricts the
    slice to rows where that variable is present, so a missing grain-size
    point (24A E02 at 14.2 / 14.6 / 14.8) drops out here and the curve simply
    joins the depths that do have data - no break, no shifted depth."""
    if df is None or not len(df): return None
    tol = SNAP_TOL_CM if snap is None else snap
    d = df.depth_cm
    ok = df[col].notna() if col is not None else pd.Series(True, index=df.index)
    idx = list(df.index[(d >= top) & (d <= base) & ok])
    up = df[(d < top) & (d >= top - tol) & ok]
    if len(up) and not _occupied(core_events, up.depth_cm.iloc[-1], eid):
        idx.insert(0, up.index[-1])
    dn = df[(d > base) & (d <= base + tol) & ok]
    if len(dn) and not _occupied(core_events, dn.depth_cm.iloc[0], eid):
        idx.append(dn.index[0])
    return df.loc[idx].sort_values('depth_cm')

def _count(df, core_events, eid, top, base, snap=None):
    if df is None: return 0
    seg = event_slice(df, core_events, eid, top, base, snap=snap)
    return 0 if seg is None else len(seg)

AUDIT = []

def build_events(core, ev, typ):
    rows, keep = [], {}
    allev = [(f'E{e:02d}', *ev[e], typ[e]) for e in sorted(ev)]
    for e in sorted(ev):
        top, base = ev[e]
        eid = f'E{e:02d}'
        n_x = _count(CORES[core], allev, eid, top, base)
        n_g = _count(GS.get(core), allev, eid, top, base)
        n_s = _count(GS.get(core), allev, eid, top, base, snap=0.0)  # strict
        ok  = {'xrf':    n_x >= MIN_SAMPLES,
               'gs':     n_g >= MIN_SAMPLES,
               'either': n_x >= MIN_SAMPLES or n_g >= MIN_SAMPLES,
               'both':   n_x >= MIN_SAMPLES and n_g >= MIN_SAMPLES,
              }[MIN_SAMPLES_BASIS]
        rows.append((f'E{e:02d}', top, base, n_x, n_g, 'keep' if ok else 'DROP', n_s))
        a = AGES_BY_CORE.get(core, {}).get(e, (np.nan, np.nan))
        AUDIT.append(dict(core=core, event=f'E{e:02d}', top_cm=top, base_cm=base,
                          age_CE=a[0], age_err=a[1], type=typ[e],
                          n_XRF=n_x, n_GS=n_g, n_GS_strict=n_s,
                          n_GS_file=N_FROM_FILE.get(core, {}).get(e),
                          status='keep' if ok else 'DROP'))
        if ok: keep[e] = (top, base)
    print(f"\n{core}   MIN_SAMPLES = {MIN_SAMPLES} on '{MIN_SAMPLES_BASIS}'")
    print(f"  {'event':6s} {'top':>7s} {'base':>7s} {'n_XRF':>6s} {'n_GS':>5s}"
          f" {'strict':>6s} {'n_file':>6s}  status")
    mism = []
    for r in rows:
        nf = N_FROM_FILE.get(core, {}).get(int(r[0][1:]))
        flag = '  no GSD (small event)' if nf == 0 else ''
        if nf and nf != r[6]:                  # compare like with like: the
            flag = '  <- differs'              # table counts strictly inside
            mism.append(f'{r[0]} ({r[6]} vs {nf})')
        print(f"  {r[0]:6s} {r[1]:7.2f} {r[2]:7.2f} {r[3]:6d} {r[4]:5d}"
              f" {r[6]:6d} {('-' if nf is None else str(nf)):>6s}  {r[5]}{flag}")
    if mism:
        print(f"  NOTE: strictly-inside grain-size counts differ from the "
              f"event table: {mism}")
        print(f"        (n_GS is what is plotted and includes boundary "
              f"snapping; 'strict' is the like-for-like comparison.)")
    dropped = [r[0] for r in rows if r[5] == 'DROP']
    print(f"  kept {len(keep)}/{len(ev)}" + (f"   dropped: {dropped}" if dropped else ""))
    return [(f'E{e:02d}', *keep[e], typ[e]) for e in sorted(keep)]

EVENTS = {'GUAC-29A': build_events('GUAC-29A', EVENTS_29A, TYPES_29A),
          'GUAC-24A': build_events('GUAC-24A', EVENTS_24A, TYPES_24A)}

AUDIT_DF = pd.DataFrame(AUDIT)
AUDIT_DF.to_csv('/content/EVENT_TABLE_current.csv', index=False)
print("\nEvent audit -> /content/EVENT_TABLE_current.csv"
      "   (edit it, re-upload it, and Cell 2 will read it back)")

# Colour is keyed to the EVENT NUMBER over both cores, so E07 is the same
# colour in 24A and in 29A - that is what makes the correlation readable.
_ALL_IDS = sorted({eid for c in EVENTS for eid,*_ in EVENTS[c]})
EVENT_COLORS = {c: {eid: PALETTE[_ALL_IDS.index(eid) % len(PALETTE)]
                    for eid,*_ in EVENTS[c]} for c in EVENTS}
print(f"\nEvent colours shared across cores: "
      + ", ".join(f"{e}" for e in _ALL_IDS))

def ecolor(core, eid, typ, fallback):
    if COLOR_BY == 'event': return EVENT_COLORS[core][eid]
    if COLOR_BY == 'type':  return TYPE_COLOR[typ]
    return fallback

# ---- depth window -----------------------------------------------------------
def dlim(core, d=None):
    """(bottom, top) for ylim - falls back to the data range if unset."""
    if core in DEPTH_LIM:
        top, bot = DEPTH_LIM[core]
    else:
        top, bot = float(d.depth_cm.min()), float(d.depth_cm.max())
    return bot, top

def check_window(core):
    if core not in DEPTH_LIM: return
    top, bot = DEPTH_LIM[core]
    bad = [f"{eid} ({t:.2f}-{b:.2f})" for eid,t,b,_ in EVENTS[core]
           if t < top or b > bot]
    print(f"{core}: window {top}-{bot} cm"
          + (f"   OUTSIDE/CLIPPED: {bad}" if bad else "   all kept events inside"))

# ---- photo column -----------------------------------------------------------
def photo_panel(ax, core, dmax, dmin):
    """PHOTO_MODE='space' leaves an empty box whose top and bottom edges are
    exactly DEPTH_LIM, so a photo dropped in PowerPoint only has to be scaled
    until its own cm marks meet the box edges."""
    ax.set_xticks([]); ax.set_xlim(0,1); ax.set_ylim(dmax, dmin)
    ax.tick_params(axis='y', which='both', length=0, labelleft=False)
    for s in ax.spines.values(): s.set_visible(False)
    if PHOTO_MODE == 'space' and PHOTO_GUIDE:
        for k in ['top','bottom','left','right']:
            sp = ax.spines[k]
            sp.set_visible(True); sp.set_linewidth(0.7)
            sp.set_color('0.6'); sp.set_linestyle((0,(3,3)))
        ax.text(0.5, (dmax+dmin)/2, f'core photo\n{dmin:g}-{dmax:g} cm',
                rotation=90, ha='center', va='center', fontsize=8, color='0.55')
    elif PHOTO_MODE == 'image':
        cfg = PHOTO_CFG.get(core, {})
        if not cfg.get('path'): return
        im = mpimg.imread(cfg['path'])
        l, r = cfg.get('crop_lr', (0,1))
        im = im[:, int(l*im.shape[1]):int(r*im.shape[1])]
        ax.imshow(im, extent=[0,1,cfg['bot_cm'],cfg['top_cm']], aspect='auto',
                  origin='upper', interpolation='lanczos', zorder=0)
        ax.set_ylim(dmax, dmin)
        for k in ['top','bottom','left','right']:
            sp = ax.spines[k]
            sp.set_visible(True); sp.set_linewidth(0.6); sp.set_color('0.25')

PANEL_W = 1.7   # inches per data panel (the clr figure has 11 panels)

def make_axes(core, n_data, height=9.0, widths=None):
    """Figure + axes with the photo column already sized. Returns (fig, data_axes).
    `widths` = one number per data panel, in units of PANEL_W (default 1)."""
    photo = PHOTO_MODE != 'none'
    n = n_data + (1 if photo else 0)
    widths = list(widths) if widths else [1] * n_data
    w = PANEL_W*sum(widths) + (PHOTO_W*PANEL_W if photo else 0) + 1.6
    fig, axes = plt.subplots(1, n, figsize=(w, height), sharey=True,
                             gridspec_kw={'width_ratios':
                                          ([PHOTO_W] if photo else []) + widths})
    axes = np.atleast_1d(axes)
    if photo:
        photo_panel(axes[0], core, *dlim(core, CORES[core]))
        return fig, list(axes[1:])
    return fig, list(axes)

# ---- axis / panel styling ---------------------------------------------------
def depth_axis(ax, dmax, dmin, major=10, minor=2):
    ax.set_ylim(dmax, dmin)
    ax.yaxis.set_major_locator(MultipleLocator(major))
    ax.yaxis.set_minor_locator(MultipleLocator(minor))
    ax.tick_params(axis='y', which='major', length=5, labelsize=9)
    ax.tick_params(axis='y', which='minor', length=2.5)
    ax.set_ylabel('Depth (cm)', fontsize=12)

def style_panel(ax, i, color, xlabel, fs=10):
    """Bertrand Fig. 15a: no box, x-axis alternating top/bottom."""
    for s in ['top','bottom','left','right']: ax.spines[s].set_visible(False)
    side = 'top' if i % 2 == 0 else 'bottom'
    ax.spines[side].set_visible(True); ax.spines[side].set_color(color)
    ax.xaxis.set_ticks_position(side); ax.xaxis.set_label_position(side)
    ax.tick_params(axis='x', colors=color, labelsize=8.5)
    ax.set_xlabel(xlabel, color=color, fontsize=fs, labelpad=4)
    ax.grid(False, which='both')

def finish_axes(axes, core):
    """Depth ticks on the leftmost data panel only."""
    d = CORES[core]
    for ax in axes:
        ax.spines['left'].set_visible(False); ax.spines['right'].set_visible(False)
    depth_axis(axes[0], *dlim(core, d))
    axes[0].spines['left'].set_visible(True)
    axes[0].tick_params(axis='y', which='both', labelleft=True)
    for ax in axes[1:]:
        ax.tick_params(axis='y', which='both', length=0, labelleft=False)

HEAD_SPACE = 0.945   # fraction of figure height left free for the core ID

def draw_group_headers(fig, axes, groups):
    """A line + name above each run of neighbouring panels in the same group,
    e.g. one 'Lithogenic (detrital)' bracket spanning Al ... Zr.
    `groups` has one name per panel ('' = no header over that panel)."""
    fig.canvas.draw()                                   # so sizes are known
    rend = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    # highest point of any panel INCLUDING its top x-axis label
    top = max(ax.get_tightbbox(rend).transformed(to_fig).y1 for ax in axes)
    runs = []                                           # [name, first, last]
    for i, g in enumerate(groups):
        if runs and runs[-1][0] == g: runs[-1][2] = i
        else: runs.append([g, i, i])
    for name, i0, i1 in runs:
        if not name: continue
        x0 = axes[i0].get_position().x0 + 0.004
        x1 = axes[i1].get_position().x1 - 0.004
        y = top + 0.01
        fig.add_artist(Line2D([x0, x1], [y, y], transform=fig.transFigure,
                              color='0.25', lw=1.1))
        fig.text((x0 + x1) / 2, y + 0.006, name, ha='center', va='bottom',
                 fontsize=10, fontweight='bold', color='0.2')
    return top

def finish_fig(fig, core, fname, subtitle='', axes=None, groups=None):
    """Reserve a strip at the top so the core ID never lands on an axis label.
    If `groups` is given, element-group headers are drawn above the panels."""
    fig.tight_layout(rect=[0, 0, 1, HEAD_SPACE if not groups else 0.87])
    top = HEAD_SPACE
    if groups:
        top = draw_group_headers(fig, axes, groups) + 0.05
    fig.text(0.004, max(0.998, top + 0.05), core, fontsize=13,
             fontweight='bold', va='top', ha='left')
    if subtitle:
        fig.text(0.004, max(0.962, top + 0.014), subtitle, fontsize=9,
                 color='0.35', va='top', ha='left')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# ---- ellipses (used by the LDA figures in Cells 8d and 9) -------------------
from matplotlib.patches import Ellipse
from scipy.stats import chi2

def conf_ellipse(ax, x, y, color, n_std=None, p=None, **kw):
    """Ellipse around points x, y. Give n_std (e.g. 1 = 1 sigma) or p (0.95)."""
    if len(x) < 3: return
    C = np.cov(x, y); vals, vecs = np.linalg.eigh(C); vals = np.clip(vals, 0, None)
    k = n_std if n_std is not None else np.sqrt(chi2.ppf(p, 2))
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse((np.mean(x), np.mean(y)), 2 * k * np.sqrt(vals[1]),
                         2 * k * np.sqrt(vals[0]), angle=angle, fill=False,
                         edgecolor=color, **kw))

# ---- drawing ----------------------------------------------------------------
def contig_boundaries(core, tol=None):
    """Depths where two kept events touch (the E05/E06/E07 stack)."""
    tol = CONTIG_TOL if tol is None else tol
    ev = sorted(EVENTS[core], key=lambda e: e[1])
    return [(ev[i][2] + ev[i+1][1])/2
            for i in range(len(ev)-1) if ev[i+1][1] - ev[i][2] <= tol]

def draw_boundaries(ax, core):
    for y in contig_boundaries(core):
        ax.axhline(y, **BOUNDARY_KW)

def eventline(ax, df, col, core, color, marker='o', ms=3.2, lw=0.6):
    """One line per event, drawn at each core's own sampling resolution.

    Missing values (for example the Malvern points you removed) are never
    filled in. With BREAK_AT_GAPS = True the line stops at a missing sample
    and starts again below it, so the reader can see there is no measurement
    there. With False the line joins straight across the gap."""
    if df is None or col not in df.columns: return
    step = resolution(df.depth_cm)            # normal spacing of THIS dataset
    for eid, top, base, typ in EVENTS[core]:
        seg = event_slice(df, EVENTS[core], eid, top, base, col=col)
        if seg is None or len(seg) < 1: continue
        c = ecolor(core, eid, typ, color)
        # markers: every real measurement
        ax.plot(seg[col], seg.depth_cm, ls='none', marker=marker, ms=ms,
                color=c, markeredgewidth=0)
        # line: split into pieces wherever a sample is missing
        dep = seg.depth_cm.to_numpy(float)
        if BREAK_AT_GAPS and np.isfinite(step):
            cuts = np.where(np.diff(dep) > GAP_FACTOR * step)[0] + 1
        else:
            cuts = []
        for piece in np.split(np.arange(len(seg)), cuts):
            ax.plot(seg[col].iloc[piece], dep[piece], lw=lw, color=c,
                    solid_capstyle='round')

def age_text(core, eid):
    """'E01  2010.5 ± 1.7' - median age with its 1-sigma uncertainty."""
    a = AGES_BY_CORE.get(core, {}).get(int(eid[1:]))
    if a is None: return eid
    yr, sd = a
    return f"{eid}  {yr:.{AGE_DEC}f} \u00b1 {sd:.{AGE_DEC}f}" if sd \
           else f"{eid}  {yr:.{AGE_DEC}f}"

def label_events(ax, core, side=None, fontsize=None, gap=None):
    """Age labels, de-overlapped downcore and coloured to match their curve."""
    side = AGE_SIDE if side is None else side
    fs   = AGE_FS   if fontsize is None else fontsize
    gap  = AGE_GAP  if gap is None else gap
    x, ha = (1.03,'left') if side == 'right' else (0.98,'right')
    top_lim, bot_lim = (DEPTH_LIM.get(core) or
                        (CORES[core].depth_cm.min(), CORES[core].depth_cm.max()))
    ev = sorted(EVENTS[core], key=lambda e: (e[1]+e[2])/2)
    ypos, last = [], None
    for eid, top, base, typ in ev:
        y = min(max((top+base)/2, top_lim+0.4), bot_lim-0.4)
        if last is not None and y - last < gap: y = last + gap
        ypos.append(y); last = y
    for (eid, top, base, typ), y in zip(ev, ypos):
        ax.text(x, y, age_text(core, eid), transform=ax.get_yaxis_transform(),
                fontsize=fs, va='center', ha=ha, fontweight='bold',
                color=ecolor(core, eid, typ, TYPE_COLOR[typ]), clip_on=False)

def tag_rows(d, core):
    """Adds event_type AND event_id so every figure - including the Cell 6
    scatter - can colour a sample by the event it belongs to."""
    def rt(dep):
        for eid, top, base, typ in EVENTS[core]:
            if top <= dep <= base: return pd.Series([typ, eid])
        return pd.Series(['bg', None])
    d[['event_type','event_id']] = d.depth_cm.apply(rt)
    print(f"{core}: {(d.event_type!='bg').sum()}/{len(d)} rows inside kept events")
    return d

for c in CORES: CORES[c] = tag_rows(CORES[c], c)
for c in CORES: check_window(c)
# ---- generic downcore figure -----------------------------------------------
# A "panel" is one column of the figure. Each is described by a small
# dictionary, so a figure is just a list of panels read left to right:
#   {'data':'gs',  'cols':['Dmean_um'], 'label':'Mean grain size (µm)',
#    'log':True,  'group':'Grain size'}
#   {'data':'xrf', 'col':'Mn_clr', 'label':'clr Mn', 'color':'#38a169',
#    'group':'Redox'}

def mean_gs_panel(show_sorting=False):
    """Mean grain size panel (plus sorting if asked)."""
    p = [dict(data='gs', cols=['Dmean_um'],
              label='Mean grain size (µm)\nFolk & Ward (1957)',
              log=True, group='Grain size')]
    if show_sorting:
        p.append(dict(data='gs', cols=['sorting_phi'], label='Sorting (phi)\nFolk & Ward (1957)',
                      log=False, group='Grain size'))
    return p

def clr_panels():
    """One panel per element, in CLR_GROUPS order, tagged with its group."""
    return [dict(data='xrf', col=f'{el}_clr', label=f'clr {el}',
                 color=ELCOL.get(el, '#1a202c'), group=group_name)
            for group_name, group_elements in CLR_GROUPS
            for el in group_elements]

def ratio_panels():
    """One panel per log-ratio in RATIOS."""
    return [dict(data='xrf', col=f'ln_{a}_{b}', label=label, color=colr,
                 group='Log-ratios')
            for a, b, label, colr in RATIOS]

WENTWORTH_BANDS = True   # grey Udden-Wentworth class bands behind mean size
GS_TICK_UM  = 10         # µm between ticks on grain-size axes: 0, 10, 20, 30 ...
GS_PANEL_CM = 4.5        # width of each grain-size panel in cm (other panels
                         # are PANEL_W inches = 4.3 cm)

def wentworth_bands(ax, core):
    """Shade every other Udden-Wentworth class and write its name, so the mean
    grain size can be read as 'fine silt', 'medium silt' ... directly."""
    lo_x, hi_x = ax.get_xlim()
    bot, top = dlim(core, CORES[core])            # depth window (bottom, top)
    for i, (name, lo, hi) in enumerate(WENTWORTH):
        a, b = max(lo, lo_x), min(hi, hi_x)
        if b <= a: continue                          # class not in view
        if i % 2 == 0:
            ax.axvspan(a, b, color='0.93', zorder=0, lw=0)
        if (np.log(b) - np.log(a)) / (np.log(hi_x) - np.log(max(lo_x, 1e-3))) > 0.12 \
                if ax.get_xscale() == 'log' else (b - a) / (hi_x - lo_x) > 0.12:
            xm = np.sqrt(a * b) if ax.get_xscale() == 'log' else (a + b) / 2
            ax.text(xm, bot - 0.01 * (bot - top), name, rotation=90, ha='center',
                    va='bottom', fontsize=6.5, color='0.45', zorder=0.5)
    ax.set_xlim(lo_x, hi_x)

def profile_figure(core, panels, fname='', subtitle=''):
    """Draw a list of panels for one core, all sharing the depth axis."""
    d, gs = CORES[core], GS.get(core)
    # skip grain-size panels if this core has no grain-size file
    panels = [p for p in panels if p['data'] == 'xrf' or
              (gs is not None and all(c in gs.columns for c in p['cols']))]
    widths = [GS_PANEL_CM / 2.54 / PANEL_W if p['data'] == 'gs' else 1
              for p in panels]
    fig, ax = make_axes(core, len(panels), widths=widths)
    for k, p in enumerate(panels):
        if p['data'] == 'gs':
            for c in p['cols']:
                eventline(ax[k], gs, c, core, GSCOL.get(c, '#1a202c'),
                          marker=GSMARK.get(c, 'o'))
            vals = pd.concat([gs[c] for c in p['cols']]).dropna()
            if any(c.endswith('_um') for c in p['cols']):
                # plain µm axis from 0, a tick every GS_TICK_UM: 0, 10, 20, 30 ...
                top_um = GS_TICK_UM * np.ceil(vals.max() / GS_TICK_UM) if len(vals) else 10
                ax[k].set_xlim(0, top_um)
                ax[k].xaxis.set_major_locator(MultipleLocator(GS_TICK_UM))
                if top_um / GS_TICK_UM > 6:           # many ticks -> turn labels
                    ax[k].tick_params(axis='x', labelrotation=90)
            else:
                ax[k].xaxis.set_major_locator(MaxNLocator(4))
            if WENTWORTH_BANDS and 'Dmean_um' in p['cols']:
                wentworth_bands(ax[k], core)
            axis_colour = '#1a202c'
        else:
            eventline(ax[k], d, p['col'], core, p['color'])
            ax[k].xaxis.set_major_locator(MaxNLocator(3))   # 3 ticks = readable
            axis_colour = p['color'] if COLOR_BY == 'element' else '#1a202c'
        draw_boundaries(ax[k], core)
        style_panel(ax[k], k, axis_colour, p['label'])
    finish_axes(ax, core)
    label_events(ax[-1], core)
    finish_fig(fig, core, fname, subtitle=subtitle, axes=ax,
               groups=[p['group'] for p in panels])

print(f"\nHelpers ready.  COLOR_BY = '{COLOR_BY}'   PHOTO_MODE = '{PHOTO_MODE}'")

## Cell 3 – Fig 1: clr profiles
Left to right: **mean grain size | Mn (redox) | lithogenic elements |
authigenic elements**, with a labelled bracket above each group.

In [ ]:
# =============================================================================
# CELL 3 - clr PROFILES
#          Mean grain size | Mn | lithogenic elements | authigenic elements
#          (element order and groups come from CLR_GROUPS in Cell 1)
# =============================================================================
SHOW_SORTING = False     # True = add a sorting panel next to mean grain size

for core in CORES:
    panels = mean_gs_panel(SHOW_SORTING) + clr_panels()
    profile_figure(core, panels,
                   fname=f'/content/{core}_Fig1_clr_meanGS.png',
                   subtitle=f'Mean grain size + clr (Bertrand et al. 2024 groups)'
                            f' | events only | colour by {COLOR_BY}')

## Cell 4 – Fig 2: log-ratio profiles
Left to right: **mean grain size | ln(Mn/Fe) | ln(Ti/Ca) | ln(Zr/K)**.

* **ln(Mn/Fe)** – redox. Mn is lost from sediment faster than Fe when oxygen is
  low, so *higher = more oxygen at the lake floor*.
* **ln(Ti/Ca)** – detrital input (Ti) against carbonate (Ca).
* **ln(Zr/K)** – grain size: Zr sits in coarse zircon grains, K in fine clays.
  Bertrand et al. recommend Zr/Rb; K is used because Rb is not in your list.

All ratios are natural logs (ln), like the paper, and are calculated from the raw counts.

In [ ]:
# =============================================================================
# CELL 4 - LOG-RATIO PROFILES
#          Mean grain size | ln(Mn/Fe) | ln(Ti/Ca) | ln(Zr/K)
#          (the ratios and their order come from RATIOS in Cell 1)
# =============================================================================
for core in CORES:
    panels = mean_gs_panel(SHOW_SORTING) + ratio_panels()
    names = '  '.join(label for _, _, label, _ in RATIOS)
    profile_figure(core, panels,
                   fname=f'/content/{core}_Fig2_ratios_meanGS.png',
                   subtitle=f'Mean grain size + {names} | events only | '
                            f'colour by {COLOR_BY}')

## Cell 5 – PCA of all samples (points coloured by event)

In [ ]:
# =============================================================================
# CELL 5 - PCA: biplots + % variance + LOADINGS + SAMPLE SCORES (saved to CSV)
#          PCA is fit on ALL rows (background included); SHOW picks what is drawn.
# =============================================================================
from sklearn.decomposition import PCA

SHOW = ['A','B','?']                 # add 'bg' to plot background samples too
MARK = {'bg':('o',12), 'A':('s',60), 'B':('^',70), '?':('o',40)}
PCA_RESULTS = {}

fig, axes = plt.subplots(1, 2, figsize=(18,8))
for ax, (core, d) in zip(axes, CORES.items()):
    cols = [f'{e}_clr' for e in ELS]
    X = d[cols].to_numpy(float); X = X - X.mean(0)
    p = PCA(n_components=min(len(ELS),5)).fit(X)
    S = p.transform(X); L = p.components_.T
    sc = np.abs(S[:,:2]).max()/np.abs(L[:,:2]).max()*0.9

    if 'bg' in SHOW:                          # background samples, small grey
        m = (d.event_type == 'bg').to_numpy()
        ax.scatter(S[m,0], S[m,1], s=12, c=TYPE_COLOR['bg'], edgecolor='none',
                   zorder=2, label=f'background (n={m.sum()})')
    for eid, top, base, typ in EVENTS[core]:  # one colour per event (Cell 2)
        if typ not in SHOW: continue
        m = (d.event_id == eid).to_numpy()
        if not m.any(): continue
        mk, sz = MARK[typ]
        ax.scatter(S[m,0], S[m,1], s=sz, marker=mk,
                   c=ecolor(core, eid, typ, TYPE_COLOR[typ]),
                   edgecolor='k', lw=0.5, alpha=0.9, zorder=3,
                   label=f'{eid} (n={m.sum()})')
    for i, el in enumerate(ELS):
        ax.arrow(0,0, L[i,0]*sc, L[i,1]*sc, head_width=0.02*sc, color='green',
                 lw=1.2, zorder=4)
        ax.text(L[i,0]*sc*1.13, L[i,1]*sc*1.13, el, fontsize=12,
                fontweight='bold', color='blue', ha='center', zorder=5)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(f'PC1 ({p.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({p.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(core, fontsize=13)
    ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=7, frameon=False)

    npc = min(4, S.shape[1]); pcn = [f'PC{i+1}' for i in range(npc)]
    print(f"\n===== {core} =====")
    print("% variance: " + ", ".join(
        f"PC{i+1} {v*100:.1f}%" for i,v in enumerate(p.explained_variance_ratio_[:npc])))
    print("\nLoadings:")
    print(pd.DataFrame(L[:,:npc], index=ELS, columns=pcn).round(3).to_string())

    scores = pd.DataFrame(S[:,:npc], columns=pcn)
    scores.insert(0, 'depth_cm', d.depth_cm.values)
    scores['event_type'] = d.event_type.values
    fn = f'/content/{core}_PCA_scores.csv'; scores.to_csv(fn, index=False)
    print(f"\nScores -> {fn}")
    PCA_RESULTS[core] = (p, scores, pd.DataFrame(L[:,:npc], index=ELS, columns=pcn))

plt.tight_layout()
fig.savefig('/content/Fig3_PCA_biplots.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()

# ---- PC1 / PC2 downcore -----------------------------------------------------
for core, (p, scores, _) in PCA_RESULTS.items():
    fig, ax = make_axes(core, 2)
    for j, a in enumerate(ax):
        eventline(a, scores, f'PC{j+1}', core, ['#2b6cb0','#c05621'][j])
        draw_boundaries(a, core)
        style_panel(a, j, '#1a202c',
                    f'PC{j+1} ({p.explained_variance_ratio_[j]*100:.1f}%)')
    finish_axes(ax, core)
    label_events(ax[-1], core)
    finish_fig(fig, core, f'/content/{core}_Fig3b_PCscores.png',
               subtitle='PC scores downcore | events only')

## Cell 6 – does ln(Zr/K) track the measured grain size?

In [ ]:
# =============================================================================
# CELL 6 - FIG 4: grain-size proxy ratio vs measured grain size, with r
#          (Bertrand Fig. 15b style). Downloads now happen in the last cell.
# =============================================================================
GS_MATCH_TOL = 0.3     # cm: max XRF <-> grain-size depth offset
_a, _b = GS_RATIO
RCOL, RLAB = f'ln_{_a}_{_b}', f'ln({_a}/{_b})'

def pair_gs_to_xrf(core, xrf_cols, tol=None, verbose=True):
    """One row per XRF sample, grain size AVERAGED over the GSD samples nearest
    to it. Returns the XRF columns you ask for plus every grain-size column.

    24A is 2 mm grain size against 5 mm XRF, so a nearest-neighbour join would
    hand the same XRF value to two or three grain-size rows and report n = 376
    when only ~150 independent XRF measurements exist. Averaging onto the XRF
    grid keeps n honest. 29A is 5 mm against 5 mm, so it pairs 1:1.
    Grain-size samples you deleted simply are not there, so nothing is
    invented for them."""
    d, gs = CORES[core], GS.get(core)
    if gs is None: return None
    tol = GS_MATCH_TOL if tol is None else tol
    xd = d.depth_cm.to_numpy(float); gd = gs.depth_cm.to_numpy(float)
    j   = np.abs(gd[:, None] - xd[None, :]).argmin(axis=1)   # nearest XRF row
    off = np.abs(gd - xd[j])
    use = off <= tol
    t = gs[use].drop(columns=['depth_cm']).copy()
    t['depth_cm'] = xd[j[use]]
    agg = t.groupby('depth_cm', as_index=False).mean(numeric_only=True)
    keep = ['depth_cm'] + list(xrf_cols) + ['event_type', 'event_id']
    out = pd.merge(agg, d[keep], on='depth_cm', how='inner')
    out = out.dropna(subset=list(xrf_cols))
    if verbose:
        print(f"{core}: {int(use.sum())} grain-size samples -> {len(out)} XRF "
              f"depths ({use.sum()/max(len(out),1):.1f} averaged per point)"
              + (f", {int((~use).sum())} unmatched beyond {tol} cm"
                 if (~use).any() else ""))
    return out

def pair_gs_xrf(core, tol=None):
    if RCOL not in CORES[core].columns: return None
    return pair_gs_to_xrf(core, [RCOL], tol)

for core, d in CORES.items():
    gs = GS.get(core)
    if gs is None: continue
    assert RCOL in d.columns, f"{RCOL} not computed - add {GS_RATIO} to RATIOS"
    mg = pair_gs_xrf(core)
    if mg is None: continue
    if len(mg) < 5: print(f"{core}: too few matches"); continue
    fig, axs = plt.subplots(1, 3, figsize=(17,5.6))
    for ax, m in zip(axs, ['Dmean_um','D50_um','D90_um']):
        if m not in mg.columns: continue
        for t, mk in [('bg','o'),('A','s'),('B','^'),('?','D')]:
            mm = (mg.event_type == t).to_numpy()
            if not mm.any(): continue
            cols = ([ecolor(core, e, t, TYPE_COLOR[t]) for e in mg.loc[mm,'event_id']]
                    if t != 'bg' else TYPE_COLOR['bg'])   # same event = same colour
            ax.scatter(mg.loc[mm, RCOL], mg.loc[mm, m], marker=mk,
                       s=(14 if t=='bg' else 45), c=cols,
                       edgecolor='none' if t=='bg' else 'k', lw=0.4)
        ok = mg[[RCOL, m]].dropna()
        r = np.corrcoef(ok[RCOL], ok[m])[0,1]
        ax.set_xlabel(RLAB); ax.set_ylabel(m.replace('_um',' (µm)'))
        ax.set_title(f'{m.split("_")[0].replace("Dmean","Mean")}:  r = {r:.2f}  (n = {len(ok)})')
        print(f"{core} {m}: r = {r:.3f}, n = {len(ok)}")
    fig.suptitle(f'{core}  |  geochemical proxy vs measured grain size', fontsize=13)
    plt.tight_layout()
    fig.savefig(f'/content/{core}_Fig4_scatter.png', dpi=300, bbox_inches='tight',
                facecolor='white')
    plt.show()

## Cells 8a–8e – event-separated PCA (your MATLAB script in Python)
Same steps and figure numbers as `PCA_Event_script.m`:

| MATLAB figure | Cell | What it shows |
|---|---|---|
| 01–04 | 8a | PCA of **background** samples: scree + broken stick, scores vs depth, biplot |
| 05–09 | 8b | PCA of **event** samples: scree, scores vs relative depth, biplots |
| 10–11 | 8c | heatmap of mean clr per event, boxplots per element |
| 12a, 12b, 13, 14 | 8d | LDA on the significant PCs, MANOVA, permutation test |
| 15–16 | 8e | Ward clustering of events in LDA space |

What changed: the data come from Cell 1 (no separate Excel file), events and
colours come from Cell 2, and depth runs downwards like a core log.

In [ ]:
# =============================================================================
# CELL 8a - EVENT-SEPARATED PCA  (Python version of PCA_Event_script.m)
#           Part A: split samples into BACKGROUND and EVENT, then PCA on the
#           background samples only  ->  MATLAB Figs 01-04
#
# What is different from the MATLAB script, and why:
#   * the data come straight from Cell 1 (clr of your raw counts), so there is
#     no separate "29A_forPCA_events.xlsx" to keep in sync
#   * the event list is the one from Cell 2, and event colours are the SAME
#     colours used in every other figure (E07 is one colour everywhere)
#   * depth is drawn DOWNWARDS on a vertical axis, like a core log
#   * it runs for every core in PCA_CORES
# =============================================================================
from sklearn.decomposition import PCA

PCA_CORES      = list(CORES)   # e.g. ['GUAC-29A'] to run one core only
SMOOTH_WINDOW  = 5             # running-mean window (samples), MATLAB smoothWindow
EV_MIN_SAMPLES = 4             # events with fewer XRF samples become background
                               # (MATLAB minSamplesPerEvent)
PC_NAMES  = ['PC1', 'PC2', 'PC3']
PC_COLORS = ['#1f4e9c', '#c0392b', '#8e44ad']
VEC_COLOR = (0.8, 0.1, 0.1)
EVPCA = {}                     # every result is stored here, one entry per core

def save(fig, core, name):
    """Save a figure as /content/<core>_EvPCA_<name>.png and show it."""
    fig.savefig(f'/content/{core}_EvPCA_{name}.png', dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.show()

def broken_stick(n):
    """Expected share of variance for each PC if variance were split at random.
    A PC is 'significant' when its eigenvalue beats this line."""
    return np.array([np.sum(1.0 / np.arange(k, n + 1)) for k in range(1, n + 1)]) / n

def run_pca(X):
    """PCA exactly like MATLAB pca(): centred, not scaled."""
    p = PCA().fit(X)
    return dict(coeff=p.components_.T,             # loadings  (elements x PCs)
                score=p.transform(X),              # scores    (samples x PCs)
                latent=p.explained_variance_,      # eigenvalues
                explained=p.explained_variance_ratio_ * 100)

def gap_pieces(depth, factor=1.5):
    """Index groups of samples with no depth gap between them
    (so a line is not drawn across a deleted interval or an event)."""
    step = np.median(np.diff(depth))
    cuts = np.where(np.diff(depth) > factor * step)[0] + 1
    return np.split(np.arange(len(depth)), cuts)

def depth_down(ax, label='Depth (cm)'):
    """Depth axis pointing down, like a core."""
    ax.invert_yaxis(); ax.set_ylabel(label)

def biplot_vectors(ax, coeff, names):
    """Loading arrows + element names, and the unit circle."""
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), '--', color='0.7', lw=1)
    for i, name in enumerate(names):
        ax.annotate('', xy=(coeff[i, 0], coeff[i, 1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='-|>', color=VEC_COLOR, lw=1.4))
        ax.text(coeff[i, 0] * 1.12, coeff[i, 1] * 1.12, name, fontsize=9,
                color=VEC_COLOR, ha='center', va='center', fontweight='bold')
    ax.plot([], [], color=VEC_COLOR, lw=1.4, label='Loadings')

def square_limits(ax, coeff):
    """Same range on both axes, big enough for arrows AND the unit circle."""
    lim = max(1.0, np.abs(coeff[:, :2]).max()) * 1.2
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.axhline(0, color='0.85', lw=0.6, zorder=0); ax.axvline(0, color='0.85', lw=0.6, zorder=0)

for core in PCA_CORES:
    d = CORES[core]
    cols  = [f'{el}_clr' for el in ELS]
    keep  = d[cols].var().to_numpy() > 0            # drop constant columns
    names = [el for el, k in zip(ELS, keep) if k]
    A     = d[[c for c, k in zip(cols, keep) if k]].to_numpy(float)
    depth = d.depth_cm.to_numpy(float)

    # ---- label every sample: 'bg' = background, 'E03' = inside event E03 ---
    label = np.array([eid if typ != 'bg' else 'bg'
                      for eid, typ in zip(d.event_id, d.event_type)], dtype=object)
    counts = pd.Series(label[label != 'bg']).value_counts()
    too_few = sorted(counts[counts < EV_MIN_SAMPLES].index)
    for eid in too_few:                               # MATLAB "reclassified"
        label[label == eid] = 'bg'
    active = [eid for eid, *_ in EVENTS[core]
              if eid in counts.index and eid not in too_few]
    top_of = {eid: t for eid, t, b, _ in EVENTS[core]}
    rel = np.array([depth[i] - top_of[l] if l != 'bg' else np.nan
                    for i, l in enumerate(label)])   # depth below event top
    bg, ev = (label == 'bg'), (label != 'bg')

    print(f"\n{'='*70}\n{core} - event-separated PCA\n{'='*70}")
    if too_few:
        print(f"Too few XRF samples (< {EV_MIN_SAMPLES}) -> background: {too_few}")
    print(f"Background samples : {bg.sum()}")
    print(f"Event samples      : {ev.sum()}")
    for eid in active:
        print(f"   {eid}: {(label == eid).sum()} samples")

    R = dict(names=names, A=A, depth=depth, label=label, rel=rel, bg=bg, ev=ev,
             active=active, events={eid: (t, b) for eid, t, b, _ in EVENTS[core]})
    EVPCA[core] = R

    # ---- background PCA ----------------------------------------------------
    P = run_pca(A[bg]); R['bgpca'] = P
    n = len(P['latent']); bs = broken_stick(n) * P['latent'].sum()
    dep_bg = depth[bg]

    # Fig 01 - scree plot + broken stick
    fig, ax = plt.subplots(2, 1, figsize=(7, 7))
    ax[0].bar(range(1, n + 1), P['explained'], color=(0.4, 0.6, 0.8))
    ax[0].set(xlabel='Principal component', ylabel='Variance explained (%)',
              title='Background PCA - variance explained')
    ax[1].plot(range(1, n + 1), P['latent'], 'b-o', label='Eigenvalues')
    ax[1].plot(range(1, n + 1), bs, 'r--o', label='Broken-stick model')
    ax[1].set(xlabel='Principal component', ylabel='Eigenvalue',
              title='Background PCA - broken-stick model'); ax[1].legend()
    fig.suptitle(core, fontweight='bold'); fig.tight_layout()
    save(fig, core, 'Fig01_bg_scree')

    # Fig 02 - background PC scores vs depth, with running mean
    fig, ax = plt.subplots(1, 3, figsize=(10, 8), sharey=True)
    for p in range(3):
        s = P['score'][:, p]
        ax[p].plot(s, dep_bg, color='0.7', lw=0.8, label=f'{PC_NAMES[p]} scores')
        ax[p].plot(pd.Series(s).rolling(SMOOTH_WINDOW, center=True,
                                        min_periods=1).mean(),
                   dep_bg, color=PC_COLORS[p], lw=2,
                   label=f'{SMOOTH_WINDOW}-point running mean')
        ax[p].set_xlabel(f'{PC_NAMES[p]} ({P["explained"][p]:.1f}%)')
        ax[p].legend(fontsize=7, loc='lower right')
    depth_down(ax[0]); fig.suptitle(f'{core} - background PCA: PC scores vs depth',
                                    fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig02_bg_scores_runmean')

    # Fig 03 - background biplot coloured by depth
    fig, ax = plt.subplots(figsize=(8, 7))
    sc = P['score'][:, :2] / np.abs(P['score'][:, :2]).max()   # scaled to +-1
    h = ax.scatter(sc[:, 0], sc[:, 1], c=dep_bg, cmap='viridis_r', s=30,
                   edgecolor='none', label='Background samples')
    cb = fig.colorbar(h, ax=ax); cb.set_label('Depth (cm)'); cb.ax.invert_yaxis()
    biplot_vectors(ax, P['coeff'], names); square_limits(ax, P['coeff'])
    ax.set(xlabel=f'PC1 ({P["explained"][0]:.1f}%)',
           ylabel=f'PC2 ({P["explained"][1]:.1f}%)',
           title=f'{core} - background PCA biplot, coloured by depth')
    ax.legend(loc='upper left', bbox_to_anchor=(1.25, 1), fontsize=8)
    fig.tight_layout(); save(fig, core, 'Fig03_bg_biplot_depth')

    # Fig 04 - background PC scores at their true core depth (gaps left open)
    fig, ax = plt.subplots(1, 3, figsize=(10, 8), sharey=True)
    for p in range(3):
        s = P['score'][:, p]
        for k, idx in enumerate(gap_pieces(dep_bg)):
            ax[p].plot(s[idx], dep_bg[idx], color='0.7', lw=0.8,
                       label=f'{PC_NAMES[p]} scores' if k == 0 else None)
        ax[p].scatter(s, dep_bg, s=12, color=PC_COLORS[p], edgecolor='none',
                      label=f'{PC_NAMES[p]} points')
        ax[p].set_xlabel(f'{PC_NAMES[p]} ({P["explained"][p]:.1f}%)')
        ax[p].legend(fontsize=7, loc='lower right')
    depth_down(ax[0]); fig.suptitle(f'{core} - background PCA: PC scores at core depth',
                                    fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig04_bg_scores_depth')

In [ ]:
# =============================================================================
# CELL 8b - EVENT-SEPARATED PCA, Part B: PCA on the EVENT samples only
#           -> MATLAB Figs 05-09
# =============================================================================
for core in PCA_CORES:
    R = EVPCA[core]
    ev, label, active = R['ev'], R['label'], R['active']
    A_ev   = R['A'][ev]
    dep_ev = R['depth'][ev]
    lab_ev = label[ev]
    rel_ev = R['rel'][ev]
    colour = EVENT_COLORS[core]            # same colours as every other figure

    P = run_pca(A_ev)
    R.update(evpca=P, A_ev=A_ev, dep_ev=dep_ev, lab_ev=lab_ev, rel_ev=rel_ev)
    n = len(P['latent']); bs = broken_stick(n) * P['latent'].sum()
    R['bs_ev'] = bs

    def ev_label(eid):
        t, b = R['events'][eid]
        return f'{eid} ({t:.1f}-{b:.1f} cm)'

    # Fig 05 - scree + broken stick
    fig, ax = plt.subplots(2, 1, figsize=(7, 7))
    ax[0].bar(range(1, n + 1), P['explained'], color=(0.8, 0.4, 0.2))
    ax[0].set(xlabel='Principal component', ylabel='Variance explained (%)',
              title='Event PCA - variance explained')
    ax[1].plot(range(1, n + 1), P['latent'], 'b-o', label='Eigenvalues')
    ax[1].plot(range(1, n + 1), bs, 'r--o', label='Broken-stick model')
    ax[1].set(xlabel='Principal component', ylabel='Eigenvalue',
              title='Event PCA - broken-stick model'); ax[1].legend()
    fig.suptitle(core, fontweight='bold'); fig.tight_layout()
    save(fig, core, 'Fig05_ev_scree')

    # Fig 06 - event PC scores vs depth BELOW EACH EVENT'S TOP
    fig, ax = plt.subplots(1, 3, figsize=(12, 7), sharey=True)
    for p in range(3):
        for eid in active:
            m = lab_ev == eid
            o = np.argsort(rel_ev[m])
            ax[p].plot(P['score'][m, p][o], rel_ev[m][o], '-o', ms=3, lw=1.4,
                       color=colour[eid], label=eid)
        ax[p].axvline(0, color='0.6', lw=0.6, ls='--')
        ax[p].set_xlabel(f'{PC_NAMES[p]} ({P["explained"][p]:.1f}%)')
    depth_down(ax[0], 'Depth below event top (cm)')
    ax[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7,
                  frameon=False)
    fig.suptitle(f'{core} - event PCA: PC scores vs relative depth', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig06_ev_scores_reldepth')

    # Fig 07 - event biplot coloured by event
    sc = P['score'][:, :2] / np.abs(P['score'][:, :2]).max()
    R['sc_ev'] = sc
    fig, ax = plt.subplots(figsize=(9, 7))
    for eid in active:
        m = lab_ev == eid
        ax.scatter(sc[m, 0], sc[m, 1], s=35, color=colour[eid],
                   edgecolor='none', label=ev_label(eid))
    biplot_vectors(ax, P['coeff'], R['names']); square_limits(ax, P['coeff'])
    ax.set(xlabel=f'PC1 ({P["explained"][0]:.1f}%)',
           ylabel=f'PC2 ({P["explained"][1]:.1f}%)',
           title=f'{core} - event PCA biplot, coloured by event')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7, frameon=False)
    fig.tight_layout(); save(fig, core, 'Fig07_ev_biplot_event')

    # Fig 08 - event biplot coloured by relative depth
    fig, ax = plt.subplots(figsize=(8, 7))
    h = ax.scatter(sc[:, 0], sc[:, 1], c=rel_ev, cmap='viridis_r', s=35,
                   edgecolor='none', label='Event samples')
    cb = fig.colorbar(h, ax=ax); cb.set_label('Depth below event top (cm)')
    cb.ax.invert_yaxis()
    biplot_vectors(ax, P['coeff'], R['names']); square_limits(ax, P['coeff'])
    ax.set(xlabel=f'PC1 ({P["explained"][0]:.1f}%)',
           ylabel=f'PC2 ({P["explained"][1]:.1f}%)',
           title=f'{core} - event PCA biplot, coloured by relative depth')
    ax.legend(loc='upper left', bbox_to_anchor=(1.25, 1), fontsize=8)
    fig.tight_layout(); save(fig, core, 'Fig08_ev_biplot_reldepth')

    # Fig 09 - event PC scores at their true core depth
    fig, ax = plt.subplots(1, 3, figsize=(12, 8), sharey=True)
    for p in range(3):
        for eid in active:
            m = lab_ev == eid
            ax[p].plot(P['score'][m, p], dep_ev[m], '-o', ms=3.5, lw=1.2,
                       color=colour[eid], label=ev_label(eid))
        ax[p].axvline(0, color='0.6', lw=0.6, ls='--')
        ax[p].set_xlabel(f'{PC_NAMES[p]} ({P["explained"][p]:.1f}%)')
    depth_down(ax[0])
    ax[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7,
                  frameon=False)
    fig.suptitle(f'{core} - event PCA: PC scores at core depth', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig09_ev_scores_depth')

In [ ]:
# =============================================================================
# CELL 8c - EVENT-SEPARATED PCA, Part C: what each event is made of
#           Fig 10 heatmap of mean clr per event, Fig 11 boxplots per element
# =============================================================================
for core in PCA_CORES:
    R = EVPCA[core]
    names, active = R['names'], R['active']
    A_ev, lab_ev = R['A_ev'], R['lab_ev']
    colour = EVENT_COLORS[core]

    # mean and spread of every element in every event
    mean_comp = np.array([A_ev[lab_ev == eid].mean(0) for eid in active])
    # z-score down each column: which event is high / low for that element
    z = (mean_comp - mean_comp.mean(0)) / mean_comp.std(0, ddof=1)

    # Fig 10 - heatmap (colour = z-score, number = actual mean clr value)
    fig, ax = plt.subplots(figsize=(1.0 * len(names) + 4, 0.45 * len(active) + 2.5))
    h = ax.imshow(z, cmap='viridis', aspect='auto')
    cb = fig.colorbar(h, ax=ax); cb.set_label('Z-score of mean clr value')
    ax.set_xticks(range(len(names)), names, rotation=45)
    ax.set_yticks(range(len(active)),
                  [f'{e}  ({R["events"][e][0]:.1f}-{R["events"][e][1]:.1f} cm)'
                   for e in active])
    for tick, eid in zip(ax.get_yticklabels(), active):
        tick.set_color(colour[eid]); tick.set_fontweight('bold')
    for i in range(len(active)):
        for j in range(len(names)):
            ax.text(j, i, f'{mean_comp[i, j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if z[i, j] < 0 else 'black')
    ax.set(xlabel='Element', ylabel='Event',
           title=f'{core} - mean clr composition per event (z-score colours)')
    fig.tight_layout(); save(fig, core, 'Fig10_heatmap')

    # Fig 11 - one boxplot panel per element, one box per event
    ncol = int(np.ceil(np.sqrt(len(names)))); nrow = int(np.ceil(len(names) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.6 * nrow))
    axs = np.atleast_1d(axs).ravel()
    for j, el in enumerate(names):
        data = [A_ev[lab_ev == eid, j] for eid in active]
        bp = axs[j].boxplot(data, patch_artist=True, widths=0.55,
                            flierprops=dict(marker='+', ms=4))
        for box, eid in zip(bp['boxes'], active):
            box.set_facecolor(colour[eid]); box.set_alpha(0.55)
            box.set_edgecolor(colour[eid])
        for med in bp['medians']: med.set_color('k')
        axs[j].set_xticks(range(1, len(active) + 1), active, rotation=90, fontsize=6)
        axs[j].set_title(el, fontsize=9); axs[j].set_ylabel('clr', fontsize=7)
        axs[j].tick_params(axis='y', labelsize=7)
    for ax in axs[len(names):]: ax.axis('off')
    fig.suptitle(f'{core} - clr element distributions by event', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig11_boxplots')

In [ ]:
# =============================================================================
# CELL 8d - EVENT-SEPARATED PCA, Part D: can the events be told apart?
#           LDA on the significant event PCs + MANOVA + permutation test
#           -> MATLAB Figs 12a, 12b, 13, 14
# =============================================================================
from scipy.linalg import eigh

N_PERMS      = 999      # permutations for the significance test
LOAD_THRESH  = 0.3      # |LDA loading| above this is highlighted in Fig 12b
ELLIPSE_P    = 0.95     # 95% confidence ellipses in Fig 12a

def scatter_matrices(X, labels, groups):
    """Between-group (Sb) and within-group (Sw) scatter, as in the MATLAB code."""
    mu = X.mean(0); nf = X.shape[1]
    Sb = np.zeros((nf, nf)); Sw = np.zeros((nf, nf))
    for g in groups:
        Xg = X[labels == g]
        if len(Xg) < 2: continue
        dg = (Xg.mean(0) - mu)[:, None]
        Sb += len(Xg) * dg @ dg.T
        dw = Xg - Xg.mean(0)
        Sw += dw.T @ dw
    return Sb, Sw

def manova(X, labels, groups):
    """One-way MANOVA, same output as MATLAB manova1:
    Wilks' lambda and a p-value for 'group means span <= d dimensions'."""
    n, p = X.shape; g = len(groups)
    Sb, Sw = scatter_matrices(X, labels, groups)
    lam = np.sort(np.real(np.linalg.eigvals(np.linalg.solve(Sw, Sb))))[::-1]
    lam = np.clip(lam[:min(p, g - 1)], 0, None)
    pvals, wilks = [], []
    for dim in range(len(lam)):
        L = np.prod(1.0 / (1.0 + lam[dim:]))
        stat = -(n - 1 - (p + g) / 2) * np.log(L)          # Bartlett's chi-square
        df = (p - dim) * (g - 1 - dim)
        pvals.append(chi2.sf(stat, df)); wilks.append(L)
    pvals = np.array(pvals)
    d_sig = int(np.argmax(pvals >= 0.05)) if (pvals >= 0.05).any() else len(pvals)
    return wilks[0], pvals, d_sig

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

for core in PCA_CORES:
    R = EVPCA[core]
    P, bs = R['evpca'], R['bs_ev']
    active, lab_ev = R['active'], R['lab_ev']
    colour = EVENT_COLORS[core]
    if len(active) < 3:
        print(f"{core}: only {len(active)} events - LDA needs at least 3. Skipped.")
        continue

    # ---- LDA input = the PCs that beat the broken-stick line ---------------
    sig = np.where(P['latent'] > bs)[0]
    if len(sig) < 2:
        sig = np.arange(2)       # need 2 axes to draw; MATLAB would stop here
        print(f"{core}: fewer than 2 significant PCs - using PC1 and PC2")
    X = P['score'][:, sig]
    print(f"\n{core}: LDA on {len(sig)} significant PCs "
          f"({', '.join(f'PC{i+1}' for i in sig)})")

    Sb, Sw = scatter_matrices(X, lab_ev, active)
    Sw = Sw + 1e-6 * np.eye(len(sig))                  # same safety term as MATLAB
    evals, W = eigh(Sb, Sw)                            # solves Sb w = lambda Sw w
    order = np.argsort(evals)[::-1]; evals, W = evals[order], W[:, order]
    LD = X @ W[:, :2]
    lda_expl = evals / evals[evals > 0].sum() * 100
    cent = np.array([LD[lab_ev == e].mean(0) for e in active])

    # ---- significance ------------------------------------------------------
    wilks, p_man, d_man = manova(X, lab_ev, active)
    obs = np.trace(Sb) / (np.trace(Sb) + np.trace(Sw))
    rng = np.random.default_rng(42)
    perm = np.empty(N_PERMS)
    for k in range(N_PERMS):
        Sb_p, Sw_p = scatter_matrices(X, rng.permutation(lab_ev), active)
        perm[k] = np.trace(Sb_p) / (np.trace(Sb_p) + np.trace(Sw_p))
    p_perm = np.mean(perm >= obs)
    print(f"MANOVA   Wilks' lambda = {wilks:.4f}   p = {p_man[0]:.4g} {stars(p_man[0])}"
          f"   significant dimensions = {d_man}")
    for dim, pv in enumerate(p_man):
        print(f"   means span <= {dim} dimension(s): p = {pv:.4g} {stars(pv)}")
    print(f"Permutation ({N_PERMS}x)  ratio = {obs:.4f}   p = {p_perm:.4f} {stars(p_perm)}")
    print("   Wilks' lambda near 0 = events well separated; near 1 = not separated")
    R.update(lda=dict(sig=sig, W=W, LD=LD, expl=lda_expl, cent=cent,
                      wilks=wilks, p_man=p_man, d_man=d_man, p_perm=p_perm))

    # Fig 12a - LDA scores with 95% ellipses and event centroids
    fig, ax = plt.subplots(figsize=(10, 7.5))
    yr = np.ptp(LD[:, 1])
    for i, eid in enumerate(active):
        m = lab_ev == eid; t, b = R['events'][eid]
        ax.scatter(LD[m, 0], LD[m, 1], s=40, color=colour[eid], edgecolor='none',
                   label=f'{eid} ({t:.1f}-{b:.1f} cm)')
        conf_ellipse(ax, LD[m, 0], LD[m, 1], colour[eid], p=ELLIPSE_P, ls='--', lw=1.4)
        ax.scatter(*cent[i], s=110, marker='D', color=colour[eid], edgecolor='k', lw=1.3)
        ax.text(cent[i, 0], cent[i, 1] + 0.02 * yr, eid, fontsize=7,
                fontweight='bold', ha='center', va='bottom')
    ax.plot([], [], 'k--', label=f'{ELLIPSE_P:.0%} confidence ellipse')
    ax.set(xlabel=f'LD1 ({lda_expl[0]:.1f}% of between-event variance)',
           ylabel=f'LD2 ({lda_expl[1]:.1f}% of between-event variance)')
    ax.set_title(f"{core} - LDA discriminant scores by event\n"
                 f"MANOVA: Wilks' Λ = {wilks:.4f}, p = {p_man[0]:.4g}  |  "
                 f"permutation p = {p_perm:.4f}", fontsize=11)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7, frameon=False)
    fig.tight_layout(); save(fig, core, 'Fig12a_LDA_scores')

    # Fig 12b - which elements drive LD1 and LD2
    load = P['coeff'][:, sig] @ W[:, :2]
    fig, axs = plt.subplots(2, 1, figsize=(9, 7))
    for ld, ax in enumerate(axs):
        v = load[:, ld]; dom = np.abs(v) > LOAD_THRESH
        ax.bar(range(len(v)), v, color=['#f2bf1a' if x else c for x, c in
               zip(dom, [('#3380cc', '#cc4d33')[ld]] * len(v))])
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.set_xticks(range(len(v)), R['names'], rotation=45)
        for j in np.where(dom)[0]:
            ax.text(j, v[j] + np.sign(v[j]) * np.ptp(v) * 0.04, f'{v[j]:.2f}',
                    ha='center', fontsize=7, fontweight='bold')
        ax.set_ylabel(f'LD{ld+1} loading')
        ax.set_title(f'LD{ld+1} - {lda_expl[ld]:.1f}% of between-event variance '
                     f'(yellow = |loading| > {LOAD_THRESH})', fontsize=10)
        print(f"   LD{ld+1} dominant: " + (', '.join(f'{R["names"][j]} ({v[j]:.3f})'
              for j in np.where(dom)[0]) or 'none above threshold'))
    fig.suptitle(f'{core} - LDA element loadings', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig12b_LDA_loadings')

    # Fig 13 - PC1 profile of every event, top of event at the top
    rel = R['rel_ev']; s1 = P['score'][:, 0]
    lim = (s1.min() * 1.15, s1.max() * 1.15)
    fig, axs = plt.subplots(1, len(active), figsize=(1.0 * len(active) + 1.5, 5),
                            sharex=True)
    for i, (eid, ax) in enumerate(zip(active, np.atleast_1d(axs))):
        m = lab_ev == eid; o = np.argsort(rel[m])
        ax.plot(s1[m][o], rel[m][o], '-o', ms=3, lw=1.4, color=colour[eid])
        ax.axvline(0, color='k', ls='--', lw=0.6, alpha=0.5)
        ax.set_ylim(rel[m].max(), 0); ax.set_xlim(lim)
        t, b = R['events'][eid]
        ax.set_title(f'{eid}\n{t:.1f}-{b:.1f}', fontsize=7, color=colour[eid])
        ax.tick_params(labelsize=6)
        if i == 0: ax.set_ylabel('Depth below event top (cm)', fontsize=8)
    fig.supxlabel('PC1 score', fontsize=9)
    fig.suptitle(f'{core} - event PC1 profiles (event top at the top)', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig13_event_PC1_profiles')

    # Fig 14 - the significance tests as pictures
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].bar(range(len(p_man)), p_man, color=(0.4, 0.6, 0.8))
    axs[0].axhline(0.05, color='r', ls='--', label='p = 0.05')
    axs[0].axhline(0.01, color='k', ls='--', label='p = 0.01')
    for dim, pv in enumerate(p_man):
        axs[0].text(dim, pv + 0.002, stars(pv), ha='center', fontweight='bold')
    axs[0].set_xticks(range(len(p_man)), [f'{i} dim' for i in range(len(p_man))])
    axs[0].set_ylim(0, max(p_man.max() * 1.3, 0.12))
    axs[0].set(xlabel='Null hypothesis: means span <= N dimensions', ylabel='p-value',
               title=f"MANOVA - Wilks' Λ = {wilks:.4f} | significant dims = {d_man}")
    axs[0].legend()
    axs[1].hist(perm, bins=40, color='0.75', edgecolor='w', label='Null distribution')
    axs[1].axvline(obs, color='r', lw=2.5, label=f'Observed = {obs:.4f}')
    axs[1].set(xlabel='Between-event variance ratio', ylabel='Count',
               title=f'Permutation test (n = {N_PERMS}) - p = {p_perm:.4f} {stars(p_perm)}')
    axs[1].legend(loc='upper left')
    fig.suptitle(f'{core} - LDA significance tests', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig14_significance')

In [ ]:
# =============================================================================
# CELL 8e - EVENT-SEPARATED PCA, Part E: which events look alike?
#           Ward clustering of the event centroids in LDA space
#           -> MATLAB Figs 15 and 16
# =============================================================================
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score

for core in PCA_CORES:
    R = EVPCA[core]
    if 'lda' not in R: continue
    L = R['lda']; active, lab_ev, LD, cent = R['active'], R['lab_ev'], L['LD'], L['cent']
    if len(active) < 4:
        print(f"{core}: fewer than 4 events - clustering skipped"); continue

    D = pdist(cent)                                # distance between event centres
    Z = linkage(D, 'ward')
    # try 2 ... 8 clusters and keep the one with the best silhouette score
    ks, sil = [], []
    for k in range(2, min(len(active) - 1, 8) + 1):
        lab = fcluster(Z, k, 'maxclust')
        if 2 <= len(set(lab)) < len(active):
            ks.append(k); sil.append(silhouette_score(cent, lab))
    best_k = ks[int(np.argmax(sil))]
    clus = fcluster(Z, best_k, 'maxclust')
    ccol = plt.cm.tab10(np.arange(10))
    R['clusters'] = dict(zip(active, clus))

    print(f"\n{core}: best number of clusters = {best_k} "
          f"(silhouette = {max(sil):.3f})")
    for c in range(1, best_k + 1):
        members = [e for e, cc in zip(active, clus) if cc == c]
        print(f"   Cluster {c}: " + ', '.join(
            f"{e} ({R['events'][e][0]:.1f}-{R['events'][e][1]:.1f} cm)" for e in members))

    def lda_by_cluster(ax, fs=7):
        """LDA scores coloured by cluster, centroids labelled with event number."""
        yr = np.ptp(LD[:, 1])
        for i, eid in enumerate(active):
            m = lab_ev == eid; c = ccol[(clus[i] - 1) % 10]
            ax.scatter(LD[m, 0], LD[m, 1], s=30, color=c, edgecolor='none')
            ax.scatter(*cent[i], s=110, marker='D', color=c, edgecolor='k', lw=1.3)
            ax.text(cent[i, 0], cent[i, 1] + 0.02 * yr, eid, fontsize=fs,
                    fontweight='bold', ha='center', va='bottom')
        for c in range(1, best_k + 1):
            members = ', '.join(e for e, cc in zip(active, clus) if cc == c)
            ax.scatter([], [], s=60, color=ccol[(c - 1) % 10], edgecolor='k',
                       label=f'Cluster {c}: {members}')
        ax.set(xlabel=f'LD1 ({L["expl"][0]:.1f}%)', ylabel=f'LD2 ({L["expl"][1]:.1f}%)',
               title='LDA scores coloured by cluster')
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=7, frameon=False)

    # Fig 15 - four-panel clustering summary
    fig, axs = plt.subplots(2, 2, figsize=(15, 11))
    dendrogram(Z, labels=active, orientation='left', ax=axs[0, 0],
               color_threshold=Z[-(best_k - 1), 2] if best_k > 1 else None)
    axs[0, 0].set(xlabel='Ward linkage distance', ylabel='Event',
                  title=f'Dendrogram (Ward) - best K = {best_k}')
    axs[0, 1].bar(ks, sil, color=(0.6, 0.7, 0.8))
    axs[0, 1].bar([best_k], [max(sil)], color=(0.9, 0.4, 0.2))
    axs[0, 1].set(xlabel='Number of clusters (K)', ylabel='Mean silhouette score',
                  title=f'Silhouette analysis - best K = {best_k} ({max(sil):.3f})')
    axs[0, 1].set_xticks(ks)
    lda_by_cluster(axs[1, 0])
    o = np.argsort(clus, kind='stable'); Dm = squareform(D)[np.ix_(o, o)]
    h = axs[1, 1].imshow(Dm, cmap='viridis')
    fig.colorbar(h, ax=axs[1, 1]).set_label('Euclidean distance in LDA space')
    lab_sorted = [active[i] for i in o]
    axs[1, 1].set_xticks(range(len(o)), lab_sorted, rotation=45, fontsize=7)
    axs[1, 1].set_yticks(range(len(o)), lab_sorted, fontsize=7)
    for b in np.cumsum(np.bincount(clus)[1:])[:-1]:
        axs[1, 1].axvline(b - 0.5, color='w', lw=2.5)
        axs[1, 1].axhline(b - 0.5, color='w', lw=2.5)
    axs[1, 1].set_title('Pairwise distance matrix (sorted by cluster)')
    fig.suptitle(f'{core} - event clustering: {best_k} clusters', fontweight='bold')
    fig.tight_layout(); save(fig, core, 'Fig15_clustering')

    # Fig 16 - the LDA-by-cluster panel on its own, larger
    fig, ax = plt.subplots(figsize=(10, 7))
    lda_by_cluster(ax, fs=8)
    ax.set_title(f'{core} - LDA scores coloured by cluster membership')
    fig.tight_layout(); save(fig, core, 'Fig16_LDA_clusters')

## Cell 9 – Bertrand et al. (2024) Fig. 12 with your data
*Paper:* a linear discriminant biplot of sediment **sources** (bedrock and soil
samples, Liu et al. 2020) with 1-sigma ellipses; river sediments (blue circles)
are projected onto it to see which source they resemble.

*Here:* each **event** plays the role of a source (ellipse in the event colour),
and **background** samples are the blue circles. The variables are the clr
elements **plus ln(mean grain size)**. Arrows show which variable pushes
samples in which direction. The title gives the leave-one-out score: how often
a sample is put back in its own event when it is left out of the fit.

In [ ]:
# =============================================================================
# CELL 9 - BERTRAND et al. (2024) FIG. 12 WITH YOUR DATA
#          Linear discriminant biplot (after Liu et al., 2020)
#
# In the paper:  groups   = sediment SOURCES (bedrock, soil samples)
#                ellipses = 1-sigma spread of each source
#                blue circles = river sediments, projected onto the plot to
#                see which source they are closest to
# Here:          groups   = your EVENTS (each event = one "source")
#                ellipses = 1-sigma spread of each event, in the event colour
#                blue circles = BACKGROUND samples, projected the same way
#                arrows   = which variable pushes samples in which direction
# Variables used: the clr elements from Cell 1 + ln(mean grain size).
# =============================================================================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, LeaveOneOut

F12_ELEMENTS   = ELS                 # clr elements to use (edit to drop some)
F12_GRAINSIZE  = ['Dmean_um']        # add 'sorting_phi', 'D90_um' ... if wanted
F12_MIN_N      = 4                   # an event needs this many samples WITH
                                     # grain size to be a group
F12_N_STD      = 1.0                 # ellipse size: 1 = 1 sigma (as in Fig. 12)
F12_SHOW_BG    = True                # project background samples (blue circles)
F12_LABEL_BG   = False               # write the depth next to each blue circle
F12_ARROWS     = True                # variable arrows (the "bi" in biplot)
BG_BLUE        = '#1f5fbf'

for core in CORES:
    if GS.get(core) is None:
        print(f"{core}: no grain-size data - Fig 12 skipped"); continue
    xcols = [f'{el}_clr' for el in F12_ELEMENTS]
    m = pair_gs_to_xrf(core, xcols, verbose=False)     # XRF + GS on one grid
    for c in F12_GRAINSIZE:                             # grain size -> log scale
        m[f'ln_{c}'] = np.log(m[c]) if c.endswith('_um') else m[c]
    feats  = xcols + [f'ln_{c}' for c in F12_GRAINSIZE]
    fnames = [f'clr {el}' for el in F12_ELEMENTS] + \
             [('ln ' if c.endswith('_um') else '') +
              c.replace('Dmean_um', 'mean GS').replace('_um', '').replace('_phi', '')
              for c in F12_GRAINSIZE]
    m = m.dropna(subset=feats)                          # only complete rows

    # which events have enough complete samples to be a group?
    n_per = m[m.event_type != 'bg'].event_id.value_counts()
    groups = [eid for eid, *_ in EVENTS[core] if n_per.get(eid, 0) >= F12_MIN_N]
    small  = [eid for eid, *_ in EVENTS[core] if 0 < n_per.get(eid, 0) < F12_MIN_N]
    if len(groups) < 3:
        print(f"{core}: only {len(groups)} events with >= {F12_MIN_N} samples - skipped")
        continue

    # standardise every variable (mean 0, sd 1) so clr and grain size compare
    mu, sd = m[feats].mean(), m[feats].std()
    Z = ((m[feats] - mu) / sd).to_numpy(float)
    is_grp = m.event_id.isin(groups).to_numpy()
    is_bg  = (m.event_type == 'bg').to_numpy()
    y = m.event_id.to_numpy()[is_grp]

    # 'svd' copes with clr data, whose values always add up to zero (that
    # makes one variable redundant, which breaks the other LDA solvers)
    lda = LinearDiscriminantAnalysis(solver='svd')
    lda.fit(Z[is_grp], y)
    S_grp = lda.transform(Z[is_grp])[:, :2]
    S_bg  = lda.transform(Z[is_bg])[:, :2]
    expl  = lda.explained_variance_ratio_ * 100

    # how well do the variables tell the events apart? (leave-one-out test)
    acc = cross_val_score(LinearDiscriminantAnalysis(solver='svd'),
                          Z[is_grp], y, cv=LeaveOneOut()).mean()
    print(f"\n{core}: {len(groups)} events, {is_grp.sum()} event samples, "
          f"{is_bg.sum()} background samples, {len(feats)} variables")
    if small: print(f"   too few complete samples, left out: {small}")
    print(f"   LD1 {expl[0]:.1f}%  LD2 {expl[1]:.1f}%   "
          f"leave-one-out classification = {acc:.0%} correct")

    # ---- plot ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 8))
    for eid in groups:
        k = (y == eid); col = EVENT_COLORS[core][eid]
        ax.scatter(S_grp[k, 0], S_grp[k, 1], s=26, marker='s', color=col,
                   edgecolor='none', alpha=0.75)
        conf_ellipse(ax, S_grp[k, 0], S_grp[k, 1], col, n_std=F12_N_STD, lw=1.8)
        cx, cy = S_grp[k].mean(0)
        ax.text(cx, cy, eid, fontsize=8.5, fontweight='bold', color=col,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))
        t, b = {e[0]: e[1:3] for e in EVENTS[core]}[eid]      # event top, base
        ax.plot([], [], color=col, lw=1.8, marker='s', ms=5,
                label=f'{eid} ({t:.1f}-{b:.1f} cm, n={k.sum()})')
    if F12_SHOW_BG and is_bg.any():
        ax.scatter(S_bg[:, 0], S_bg[:, 1], s=42, facecolor='none',
                   edgecolor=BG_BLUE, lw=1.2, zorder=4,
                   label=f'Background samples, projected (n={is_bg.sum()})')
        if F12_LABEL_BG:
            for (sx, sy), dep in zip(S_bg, m.depth_cm.to_numpy()[is_bg]):
                ax.text(sx, sy, f' {dep:g}', fontsize=6, color=BG_BLUE)
    if F12_ARROWS:
        # arrow = correlation of each variable with LD1 and LD2, scaled to fit
        r = np.array([[np.corrcoef(Z[is_grp][:, j], S_grp[:, i])[0, 1]
                       for i in range(2)] for j in range(len(feats))])
        # longest arrow = 85% of the event cloud's smaller half-width
        scale = 0.85 * np.abs(S_grp).max(0).min() / np.abs(r).max()
        tips = np.vstack([r * scale * 1.2, [[0, 0]]])
        ax.scatter(tips[:, 0], tips[:, 1], s=0)          # makes room for labels
        for j, name in enumerate(fnames):
            gs_var = j >= len(F12_ELEMENTS)
            ac = '#111111' if gs_var else '0.35'
            ax.annotate('', xy=(r[j, 0] * scale, r[j, 1] * scale), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='-|>', color=ac,
                                        lw=1.6 if gs_var else 1.0))
            ax.text(r[j, 0] * scale * 1.1, r[j, 1] * scale * 1.1, name,
                    fontsize=8.5, color=ac, ha='center', va='center',
                    fontweight='bold' if gs_var else 'normal')
    ax.axhline(0, color='0.85', lw=0.6, zorder=0); ax.axvline(0, color='0.85', lw=0.6, zorder=0)
    ax.set_xlabel(f'LD1 ({expl[0]:.1f}%)', fontsize=11)
    ax.set_ylabel(f'LD2 ({expl[1]:.1f}%)', fontsize=11)
    ax.set_title(f'{core} - linear discriminant biplot of events '
                 f'(clr XRF + grain size)\nellipses = {F12_N_STD:g} sigma  |  '
                 f'leave-one-out correct = {acc:.0%}', fontsize=11)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7.5,
              frameon=False)
    fig.tight_layout()
    fig.savefig(f'/content/{core}_Fig12_LDA_biplot_Bertrand.png', dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.show()

## Cell 10 – Bertrand et al. (2024) Fig. 20A  *(to do)*
Waiting for the paper, or a screenshot of Fig. 20 and its caption.

In [ ]:
# =============================================================================
# CELL 10 - BERTRAND et al. (2024) FIG. 20A WITH YOUR DATA   <<< NOT BUILT YET >>>
#
# The paper could not be opened from the environment that wrote this notebook,
# so the layout of Fig. 20A is still unknown. Send the PDF (or a screenshot of
# Fig. 20 and its caption) and this cell will be filled in. It will use the
# same event colours as everything else: EVENT_COLORS[core][event_id].
# =============================================================================
print("Fig 20A: waiting for the paper / a screenshot of Fig. 20 - nothing drawn yet.")

## Cell 7 – QC: run before using any figure

In [ ]:
# =============================================================================
# CELL 7 - QC. Independent checks on everything the figures rest on.
#          Run this before any poster, presentation or submission.
#          Writes /content/QC_report.txt. "FAIL" means do not use the figures.
# =============================================================================
import io
rep, nfail = io.StringIO(), 0

def chk(name, ok, detail=''):
    global nfail
    tag = 'PASS' if ok else 'FAIL'
    if not ok: nfail += 1
    line = f"[{tag}] {name}" + (f"  -  {detail}" if detail else '')
    print(line); rep.write(line + "\n")

print("="*78); rep.write("Lachua pipeline QC\n" + "="*78 + "\n")

# 1. clr identity: clr(A) - clr(B) must equal ln(A/B) exactly ------------------
#    The two are computed by different routes (geometric-mean subtraction vs a
#    direct ratio of raw counts), so agreement means both are right.
for core, d in CORES.items():
    worst = 0
    for a, b, lab, _ in RATIOS:
        if f'{a}_clr' in d and f'{b}_clr' in d:
            worst = max(worst, float(np.abs((d[f'{a}_clr'] - d[f'{b}_clr'])
                                            - d[f'ln_{a}_{b}']).max()))
    chk(f"{core} log-ratios agree with clr", worst < 1e-9, f"max |diff| = {worst:.2e}")

# 2. depths -------------------------------------------------------------------
for core, d in CORES.items():
    chk(f"{core} XRF depths unique & increasing",
        d.depth_cm.is_unique and d.depth_cm.is_monotonic_increasing)
    gs = GS.get(core)
    if gs is not None:
        chk(f"{core} grain-size depths unique & increasing",
            gs.depth_cm.is_unique and gs.depth_cm.is_monotonic_increasing)
        chk(f"{core} grain-size depths overlap XRF range",
            gs.depth_cm.min() >= d.depth_cm.min() - 1 and
            gs.depth_cm.max() <= d.depth_cm.max() + 1,
            f"GS {gs.depth_cm.min():.2f}-{gs.depth_cm.max():.2f} vs "
            f"XRF {d.depth_cm.min():.2f}-{d.depth_cm.max():.2f} cm")

# 3. grain-size percentiles ---------------------------------------------------
for core, gs in GS.items():
    if gs is None: continue
    if {'D10_um','D50_um','D90_um'} <= set(gs.columns):
        bad = int(((gs.D10_um > gs.D50_um) | (gs.D50_um > gs.D90_um)).sum())
        chk(f"{core} D10 <= D50 <= D90", bad == 0, f"{bad} rows out of order")
    if 'Dmean_um' in gs:
        out = int(((gs.Dmean_um < gs.D10_um) | (gs.Dmean_um > gs.D90_um)).sum())
        chk(f"{core} mean sits between D10 and D90", out == 0, f"{out} rows outside")
    if 'sorting_phi' in gs:
        s = gs.sorting_phi.dropna()
        chk(f"{core} sorting positive & plausible",
            bool((s > 0).all() and (s < 8).all()),
            f"range {s.min():.2f}-{s.max():.2f} phi")

# 4. events vs what is actually plotted ---------------------------------------
for core in CORES:
    ev = sorted(EVENTS[core], key=lambda e: e[1])
    over = [(ev[i][0], ev[i+1][0]) for i in range(len(ev)-1)
            if ev[i+1][1] < ev[i][2] - 1e-9]
    chk(f"{core} no overlapping events", not over, str(over))
    win = DEPTH_LIM.get(core)
    if win:
        out = [e[0] for e in ev if e[1] < win[0] or e[2] > win[1]]
        chk(f"{core} all events inside plot window {win}", not out, str(out))
    worst = 0.0
    for eid, top, base, typ in ev:
        seg = event_slice(CORES[core], EVENTS[core], eid, top, base)
        if seg is None or not len(seg): continue
        worst = max(worst, top - seg.depth_cm.min(), seg.depth_cm.max() - base)
    chk(f"{core} snapping stayed within SNAP_TOL_CM", worst <= SNAP_TOL_CM + 1e-9,
        f"largest reach outside a boundary = {worst:.2f} cm")

# 5. colours are shared across cores -------------------------------------------
shared = {e for e in EVENT_COLORS.get('GUAC-24A', {})} & \
         {e for e in EVENT_COLORS.get('GUAC-29A', {})}
mism = [e for e in shared
        if EVENT_COLORS['GUAC-24A'][e] != EVENT_COLORS['GUAC-29A'][e]]
chk("same event number = same colour in both cores", not mism, str(mism))

# 6. ages ----------------------------------------------------------------------
for core in CORES:
    ev = sorted(EVENTS[core], key=lambda e: e[1])
    ages = [(eid, AGES_BY_CORE.get(core, {}).get(int(eid[1:]), (np.nan,))[0])
            for eid, *_ in ev]
    have = [(e, a) for e, a in ages if np.isfinite(a)]
    chk(f"{core} every plotted event has an age",
        len(have) == len(ages), f"missing: {[e for e,a in ages if not np.isfinite(a)]}")
    inv = [have[i][0] for i in range(len(have)-1) if have[i+1][1] > have[i][1]]
    chk(f"{core} ages get older downcore", not inv, f"reversals at {inv}")

# 7. hand-check table ----------------------------------------------------------
#    Verify a few of these in Excel: ln(A/B) = LN(A/B) on the RAW counts.
rep.write("\nSpot check - recompute these by hand from the raw file:\n")
print("\nSpot check (verify in Excel: ln(A/B) = LN(raw A / raw B)):")
a, b = GS_RATIO
for core, d in CORES.items():
    smp = d.sample(min(3, len(d)), random_state=0).sort_values('depth_cm')
    for _, r in smp.iterrows():
        line = (f"  {core} {r.depth_cm:7.2f} cm   {a}={r[a]:10.0f}  {b}={r[b]:10.0f}"
                f"   ln({a}/{b}) = {r[f'ln_{a}_{b}']:+.4f}")
        print(line); rep.write(line + "\n")

# 8. figure inventory ----------------------------------------------------------
pngs = sorted(os.path.basename(f) for f in glob.glob('/content/*.png'))
chk("expected figure count", len(pngs) >= 4*len(CORES) + 1, f"{len(pngs)} PNGs written")

print("\n" + "="*78)
verdict = (f"{nfail} FAILED check(s) - do NOT use these figures until fixed"
           if nfail else "All checks passed.")
print(verdict); rep.write("\n" + verdict + "\n")
rep.write("""
Checks a human still has to do:
 1. Open one figure next to the core photo. Every event band must line up with
    a visible deposit. If a curve sits above or below its band, the depth
    registration is wrong.
 2. Take two D50 values from the printed spot check depths and compare them
    against the Malvern report for those samples.
 3. Compare the plotted ages against your age-depth model output.
 4. Check EVENT_TABLE_current.csv: n_XRF and n_GS are the sample counts you
    will quote in the thesis. Any event with n_XRF close to MIN_SAMPLES is
    weakly supported and should be described as such.
 5. Have your advisor look at the event list and the A/B assignments before
    anything goes on a poster.
""")
open('/content/QC_report.txt','w').write(rep.getvalue())
print("\nSaved /content/QC_report.txt")
try: files.download('/content/QC_report.txt')
except Exception: pass

## Cell 11 – download all figures and tables

In [ ]:
# =============================================================================
# CELL 11 - DOWNLOAD every figure (PNG), the PCA scores and the event table
# =============================================================================
out = (sorted(glob.glob('/content/*.png')) +
       sorted(glob.glob('/content/*PCA_scores.csv')) +
       sorted(glob.glob('/content/GS_STATS_*.csv')) +
       ['/content/EVENT_TABLE_current.csv'])
print(f"{len(out)} files to download")
for f in out:
    if os.path.exists(f):
        files.download(f)